# Dados abertos CAPES - Modelagem para painel do GID

In [2]:
##DESCOMENTE A LINHA ABAIXO NA PRIMEIRA VEZ QUE FOR RODAR O SCRIPT
#%pip install sqlcipher3 dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
##SEMPRE EXECUTAR ESSA CÉLULA!!!!

#Importando bibliotecas necessárias:
import os
import time
import re
import ssl
import requests
from urllib.parse import urlparse
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import json
import numpy as np
import hashlib
from sqlcipher3 import dbapi2 as sqlite # No linux, a instalação do pysqlcipher3 pode envolver a instalação de bibliotecas como libsqlcipher-dev no sistema
from dotenv import load_dotenv
from concurrent.futures import ProcessPoolExecutor


#Diretórios usados para armazenar os dados (baixados e processados)
dirs = {
    'download_dir': 'capes_csv_files',
    'filtered_dir': 'ufrj_data',
    'processed_dir': 'sucupira_painel',
    'discentes': 'discentes',
    'docentes': 'docentes',
    'programas': 'programas',
    'cursos':  'programas',
    'producao': 'producao',
    'producao_detalhe': 'producao',
    'producao_autor': 'producao',
    'projetos': 'projetos',
    'membros': 'projetos',
    'financiadores': 'financiadores',
    'btd': 'btd',
}

#Base directories
download_dir = dirs.get('download_dir')
filtered_dir = dirs.get('filtered_dir')
processed_dir = dirs.get('processed_dir')

## Download dos dados abertos da CAPES

In [4]:
# Configurações do download
api_url = "https://dadosabertos.capes.gov.br/api/3/action/package_search"
organization = "diretoria-de-avaliacao"
output_dir = dirs.get('download_dir', 'capes_csv_files')
timeout_seconds = 30
max_retries = 3


prefix_substring_dirs = {
    "ddi-br-capes-colsucup-": {
        "projeto-financiador": dirs.get("financiadores", "financiadores"),
        "projeto": dirs.get("projetos", "projetos"),
    }, 
    "br-capes-colsucup-": {
        "prod": dirs.get("producao", "producao"),
        "producao": dirs.get("producao", "producao"),
        "projeto": dirs.get("projetos", "projetos"),
        "membro": dirs.get("projetos", "projetos"),
        "prog": dirs.get("programas", "programas"),
        "curso": dirs.get("cursos", "cursos"),
		"discentes": dirs.get("discentes", "discentes"),
		"docente": dirs.get("docentes", "docentes"),
        "financiador": dirs.get("financiadores", "financiadores"),
    },
    "br-capes-col-": {
        "proj": dirs.get("projetos", "projetos"),
        "producao": dirs.get("producao", "producao"),
        "prod": dirs.get("producao", "producao"),
    },
	"br-colsucup-": {
		"prod": dirs.get("producao", "producao"),
	},
    "br-capes-btd-": {
        "": dirs.get("btd", "btd")
    },
} # Mapeia prefix+substring para pastas (diretórios)

# Sessão com retry
session = requests.Session()
retry = Retry(total=max_retries, backoff_factor=1)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

# Contexto SSL customizado (caso precise ignorar erros SSL)
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

# Criar diretório base
os.makedirs(output_dir, exist_ok=True)

def sanitize_filename(filename):
    """Remove caracteres inválidos para nomes de arquivo"""
    return re.sub(r'[\\/*?:"<>|]', "_", filename)

def get_with_retry(url):
    """Faz uma requisição HTTP com retry e fallback para SSL desabilitado ou HTTP"""
    for attempt in range(max_retries):
        try:
            print(f"Tentativa {attempt + 1} para {url}")
            try:
                response = session.get(url, timeout=timeout_seconds)
                response.raise_for_status()
                return response
            except requests.exceptions.SSLError:
                print("Falha SSL, tentando com verificação desativada...")
                response = session.get(url, timeout=timeout_seconds, verify=False)
                response.raise_for_status()
                return response
        except requests.exceptions.RequestException as e:
            print(f"Erro na tentativa {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
    if url.startswith('https://'):
        http_url = url.replace('https://', 'http://', 1)
        print(f"Tentando fallback para HTTP: {http_url}")
        try:
            response = session.get(http_url, timeout=timeout_seconds)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            print(f"Falha no fallback HTTP: {e}")
    return None

def get_all_csv_links_from_ckan():
    """Consulta a API CKAN e retorna todos os arquivos CSV da organização"""
    print("Consultando a API CKAN da CAPES...")
    params = {
        "fq": f"organization:{organization}",
        "rows": 1000
    }
    try:
        response = session.get(api_url, params=params, timeout=timeout_seconds)
        response.raise_for_status()
        results = response.json()["result"]["results"]
        csv_links = []
        for dataset in results:
            for resource in dataset.get("resources", []):
                if resource.get("format", "").lower() == "csv":
                    url = resource.get("url")
                    if url:
                        csv_links.append(url)
        return sorted(set(csv_links))
    except Exception as e:
        print(f"Erro ao consultar CKAN: {e}")
        return []

def detect_subfolder(filename):
    """Detecta subpasta com base nas substrings do nome do arquivo"""
    name = filename.lower()
    for prefix, substrings in prefix_substring_dirs.items():
        for substr, folder in substrings.items():
            target = prefix + substr
            if target in name:
                return folder
    return "outros"


def download_csv_files(links):
    """Baixa todos os arquivos CSV e organiza em subpastas por tipo"""
    total = len(links)
    print(f"\nIniciando download de {total} arquivos...")

    for i, url in enumerate(links, 1):
        try:
            raw_filename = url.split('/')[-1].split('?')[0]
            filename = sanitize_filename(raw_filename)
            subfolder = detect_subfolder(filename)

            subdir_path = os.path.join(output_dir, subfolder)
            os.makedirs(subdir_path, exist_ok=True)

            filepath = os.path.join(subdir_path, filename)

            if os.path.exists(filepath):
                print(f"[{i}/{total}] Já existe: {subfolder}/{filename}")
                continue

            print(f"[{i}/{total}] Baixando: {filename} para {subfolder}/")

            start_time = time.time()
            response = get_with_retry(url)
            if not response:
                print(f"Falha ao baixar {url}")
                continue

            with open(filepath, 'wb') as f:
                f.write(response.content)

            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            print(f"Salvo como {subfolder}/{filename} ({size_mb:.2f} MB) em {time.time() - start_time:.2f}s")

        except Exception as e:
            print(f"Erro ao baixar {url}: {e}")

In [5]:
#Código para download dos dados abertos - Processo potencialmente demorado
print("Iniciando processo para baixar todos os CSVs da Diretoria de Avaliação...")
start_time = time.time()
try:
    csv_links = get_all_csv_links_from_ckan()
    print(f"\n{len(csv_links)} arquivos CSV encontrados:")
    for i, link in enumerate(csv_links[:10], 1):
        print(f"{i}. {link.split('/')[-1].split('?')[0]}")
    if len(csv_links) > 10:
        print(f"... mais {len(csv_links) - 10} arquivos")
    if csv_links:
        download_csv_files(csv_links)
    print(f"\nConcluído em {time.time() - start_time:.2f}s")
except KeyboardInterrupt:
    print("\nProcesso interrompido pelo usuário.")
except Exception as e:
    print(f"Erro inesperado: {e}")

Iniciando processo para baixar todos os CSVs da Diretoria de Avaliação...
Consultando a API CKAN da CAPES...

443 arquivos CSV encontrados:
1. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-cursocdu.csv
2. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-progrtv.csv
3. br-capes-colsucup-producao-2013a2016-2017-11-01-bibliografica-tradu.csv
4. br-capes-colsucup-producao-2013a2016-2017-11-01-artistica-visual.csv
5. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-destec.csv
6. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-patente.csv
7. br-capes-colsucup-producao-2013a2016-2017-11-01-artistica-outra.csv
8. br-capes-colsucup-producao-2013a2016-2017-11-01-bibliografica-livro.csv
9. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-dprodu.csv
10. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-deapli.csv
... mais 433 arquivos

Iniciando download de 443 arquivos...
[1/443] Já existe: producao/br-capes-colsucup-producao-2013a2016-2017-11-01-tecni

## Filtrando apenas registros da UFRJ dos dados abertos CAPES - 2013 em diante

In [6]:
#Função para selecionar documentos csv via expressões regulares e após um dado ano (e.g. 2013)
#A identificação do ano através do nome de arquivo desta função está adaptada à padronização de nomes dos arquivos da CAPES
#Valores de anos que iniciam quadriênios (2013, 2017, 2021) funcionarão normalmente. Outros anos poderão apresentar problemas
def filtrar_csvs_por_diretorio_regex_e_ano(
    caminho_do_diretorio,
    padroes_regex=None,
    ano_minimo=None, 
    busca_recursiva=False
):
    """
    Lista arquivos CSV em um diretório e os filtra com base em:
    1. Padrões de expressão regular (regex)
    2. Um ano mínimo encontrado no nome do arquivo.

    Args:
        caminho_do_diretorio (str): O caminho para o diretório base onde os CSVs estão.
        padroes_regex (list, optional): Uma lista de strings de expressões regulares.
                                        Arquivos devem corresponder a *qualquer um* desses padrões.
                                        Default é None (não aplica filtro regex).
        ano_minimo (int, optional): O ano mínimo (4 dígitos) para filtrar.
                                    Captura o primeiro grupo de 4 dígitos encontrado no nome do arquivo.
                                    Default é None (não aplica filtro de ano).
        busca_recursiva (bool, optional): Se True, a função buscará CSVs em subpastas também.
                                          Default é False.

    Returns:
        list: Uma lista de caminhos completos para os arquivos CSV que atendem a todos os critérios.

    Raises:
        ValueError: Se 'caminho_do_diretorio' não for encontrado ou se nenhum filtro for especificado.
    """
    if not os.path.exists(caminho_do_diretorio):
        print(f"Erro: O diretório '{caminho_do_diretorio}' não foi encontrado.")
        return []

    if not padroes_regex and ano_minimo is None:
        print("Aviso: Nenhum filtro (regex ou ano mínimo) foi especificado. Retornando todos os CSVs.")

    todos_csvs_encontrados = []

    # 1. Obter todos os arquivos CSV do diretório (e subpastas, se recursivo)
    if busca_recursiva:
        for root, _, files in os.walk(caminho_do_diretorio):
            for nome_arquivo in files:
                if nome_arquivo.lower().endswith(".csv"):
                    todos_csvs_encontrados.append(os.path.join(root, nome_arquivo))
    else:
        for nome_arquivo in os.listdir(caminho_do_diretorio):
            caminho_completo = os.path.join(caminho_do_diretorio, nome_arquivo)
            if os.path.isfile(caminho_completo) and nome_arquivo.lower().endswith(".csv"):
                todos_csvs_encontrados.append(caminho_completo)

    arquivos_filtrados_parcial = todos_csvs_encontrados

    # 2. Aplicar filtro por padrões Regex (se fornecidos)
    if padroes_regex:
        final_regex_filter = []
        regexes_compilados = [re.compile(p, re.IGNORECASE) for p in padroes_regex]

        for caminho_completo in arquivos_filtrados_parcial:
            nome_base = os.path.basename(caminho_completo)
            for regex in regexes_compilados:
                if regex.search(nome_base):
                    final_regex_filter.append(caminho_completo)
                    break
        arquivos_filtrados_parcial = final_regex_filter

    # 3. Aplicar filtro por ano mínimo (se fornecido)
    if ano_minimo is not None:
        final_year_filter = []
        # Regex para qualquer sequência de 4 dígitos - adaptado à padronização CAPES
        # O padrão (\d{4}) captura o ano. Valores de anos que iniciam quadriênios (2013, 2017, 2021)
        # funcionarão normalmente. Outros anos poderão apresentar problemas se o formato 'discentes-AAAA'
        # não for o primeiro e único lugar onde o ano pode estar.
        padrao_ano_capes = re.compile(r'(\d{4})') 

        for caminho_completo in arquivos_filtrados_parcial:
            nome_base = os.path.basename(caminho_completo)
            match = padrao_ano_capes.search(nome_base)
            
            if match:
                ano_str = match.group(1)
                try:
                    ano_int = int(ano_str)
                    if ano_int >= ano_minimo:
                        final_year_filter.append(caminho_completo)
                except ValueError:
                    print(f"Aviso: Não foi possível converter '{ano_str}' em ano inteiro para '{nome_base}'. Ignorando.")
        arquivos_filtrados_parcial = final_year_filter

    return arquivos_filtrados_parcial

In [7]:
#Função para obter caminhos utilizando o dicionário de diretórios definido no começo deste documento
def obter_caminho_completo(basedir_key, subdir_key, dirs=dirs):
    """
    Concatena os caminhos correspondentes a 'basedir_key' e 'subdir_key'
    do dicionário 'dirs' para formar um caminho completo.

    Args:
        basedir_key (str): A chave do diretório base em 'dirs'.
        subdir_key (str): A chave do subdiretório em 'dirs'.
        dirs (dict): Dicionário com o nome dos diretórios. Default: dirs.

    Returns:
        str: O caminho completo concatenado.

    Raises:
        ValueError: Se 'basedir_key' ou 'subdir_key' não forem chaves válidas em 'dirs'.
    """
    chaves_disponiveis = list(dirs.keys())

    if basedir_key not in dirs:
        raise ValueError(
            f"Erro: Chave '{basedir_key}' não encontrada em 'dirs' para basedir. "
            f"Chaves disponíveis: {chaves_disponiveis}"
        )

    if subdir_key not in dirs:
        raise ValueError(
            f"Erro: Chave '{subdir_key}' não encontrada em 'dirs' para subdir. "
            f"Chaves disponíveis: {chaves_disponiveis}"
        )
    
    # Obtém os valores de diretório do dicionário
    base_path = dirs[basedir_key]
    sub_path = dirs[subdir_key]

    # Concatena os caminhos usando os.path.join para compatibilidade entre sistemas
    caminho_final = os.path.join(base_path, sub_path)
    
    return caminho_final

In [8]:
#Junção das funções 'filtrar_csvs_por_diretorio_regex_e_ano()' e 'obter_caminho_completo()'
#Usada para obter uma lista com os arquivos csv de interesse a serem unificados em uma tabela única
def selecionar_csvs_capes(
    basedir_key,
    subdir_key,
    dirs=dirs,
    padroes_regex=None,
    ano_minimo=2013,
    busca_recursiva=False
):
    """
    Integra as funções para obter o caminho completo do diretório e filtrar arquivos CSV.

    Args:
        dirs (dict): Dicionário com o nome dos diretórios. Default: dirs.
        basedir_key (str): A chave do diretório base em 'dirs' (e.g., 'download_dir', 'ufrj_dir').
        subdir_key (str): A chave do subdiretório em 'dirs' (e.g., 'discentes', 'producao').
        padroes_regex (list, optional): Uma lista de strings de expressões regulares para filtrar nomes de arquivo.
                                        Arquivos devem corresponder a *qualquer um* desses padrões.
                                        Default é None (não aplica filtro regex).
        ano_minimo (int, optional): O ano mínimo (4 dígitos) para filtrar.
                                    Adapta-se ao padrão "discentes-AAAA" dos arquivos da CAPES.
                                    Valores de anos que iniciam quadriênios (2013, 2017, 2021) funcionarão
                                    normalmente. Outros anos poderão apresentar problemas se o formato não se encaixar.
                                    Default é None (não aplica filtro de ano).
        busca_recursiva (bool, optional): Se True, a função buscará CSVs em subpastas do diretório gerado.
                                          Default é False.

    Returns:
        list: Uma lista de caminhos completos para os arquivos CSV que atendem a todos os critérios.

    Raises:
        ValueError: Se 'basedir_key' ou 'subdir_key' forem inválidas, ou se o diretório final não existir.
    """
    try:
        # 1. Obter o caminho completo do diretório usando as chaves
        caminho_do_diretorio_completo = obter_caminho_completo(basedir_key, subdir_key)
        print(f"Buscando arquivos no diretório: {caminho_do_diretorio_completo}")

    except ValueError as e:
        print(f"Erro ao obter caminho do diretório: {e}")
        return []

    # 2. Filtrar os arquivos CSV dentro do diretório gerado
    arquivos_selecionados = filtrar_csvs_por_diretorio_regex_e_ano(
        caminho_do_diretorio=caminho_do_diretorio_completo,
        padroes_regex=padroes_regex,
        ano_minimo=ano_minimo,
        busca_recursiva=busca_recursiva
    )

    return arquivos_selecionados

In [9]:
#Função para processar e salvar csvs relacionados em um único arquivo
def fundir_lista_csvs(
        lista_caminhos_csv,
        colunas_desejadas,
        diretorio_saida,
        condicao_filtro_funcao = None,
        nome_arquivo_saida="saida_otimizada_lista.csv",
        chunk_size=10000,
        colunas_para_int64=None
        ):
    """
    Processa uma lista de arquivos CSV, extraindo colunas e linhas específicas de forma otimizada para RAM
    usando leitura em chunks. Inclui opção para converter colunas para tipo Int64 (inteiro com nulos)
    e lida com colunas ausentes em arquivos CSV.

    Args:
        lista_caminhos_csv (list): Uma lista de strings, onde cada string é o caminho completo para um arquivo CSV.
        colunas_desejadas (list): Uma lista de nomes de colunas a serem extraídas.
                                   A coluna usada para o filtro (se houver) deve ser incluída explicitamente aqui
                                   se você quiser que ela apareça no resultado final.
        condicao_filtro_funcao (function): Uma função que recebe uma linha (como Series do pandas)
                                           e retorna True se a linha deve ser incluída, False caso contrário.
                                           Default: None (sem filtragem)
        diretorio_saida (str): O caminho para o diretório onde o arquivo de saída será salvo.
        nome_arquivo_saida (str): O nome do arquivo CSV de saída (ex: "meu_arquivo.csv").
        chunk_size (int): O número de linhas a serem lidas por vez de cada arquivo CSV.
        colunas_para_int64 (list, optional): Uma lista de nomes de colunas que devem ser convertidas
                                              para o tipo 'Int64' (inteiro com suporte a nulos) antes de salvar.
                                              Isso evita a adição de '.0' em IDs. Default é None.
    """
    primeiro_arquivo = True
    coluna_filtro = None # Manter para compatibilidade, mas não será mais usado para adicionar/remover colunas

    caminho_completo_saida = os.path.join(diretorio_saida, nome_arquivo_saida)

    if diretorio_saida and not os.path.exists(diretorio_saida):
        os.makedirs(diretorio_saida, exist_ok=True)
        print(f"Diretório de saída criado: {diretorio_saida}")

    # A lógica de modificação de colunas_desejadas e colunas_para_salvar foi removida
    colunas_para_ler = list(colunas_desejadas) # Agora 'colunas_para_ler' é simplesmente 'colunas_desejadas'
    colunas_para_salvar = list(colunas_desejadas) # E 'colunas_para_salvar' também

    # A lógica de remoção da coluna de filtro também foi removida
    # if remove_filter_col e a lógica associada não estão mais presentes

    for caminho_completo_arquivo_entrada in lista_caminhos_csv:
        if not os.path.exists(caminho_completo_arquivo_entrada):
            print(f"Aviso: Arquivo não encontrado - {caminho_completo_arquivo_entrada}. Pulando...")
            continue
        if not caminho_completo_arquivo_entrada.lower().endswith(".csv"):
            print(f"Aviso: Ignorando arquivo não CSV - {caminho_completo_arquivo_entrada}.")
            continue

        print(f"Processando {caminho_completo_arquivo_entrada}...")

        # Ler o cabeçalho para verificar as colunas presentes
        try:
            df_header = pd.read_csv(caminho_completo_arquivo_entrada, sep=';', encoding='latin1', nrows=0)
            colunas_presentes_no_arquivo = df_header.columns.tolist()
        except Exception as e:
            print(f"Erro ao ler o cabeçalho do arquivo {caminho_completo_arquivo_entrada}: {e}. Pulando...")
            continue

        # Identificar colunas a serem lidas que realmente existem no arquivo
        colunas_a_realmente_ler = [col for col in colunas_para_ler if col in colunas_presentes_no_arquivo]
        colunas_ausentes_neste_arquivo = [col for col in colunas_para_ler if col not in colunas_presentes_no_arquivo]

        if colunas_ausentes_neste_arquivo:
            print(f"Aviso: As seguintes colunas desejadas não foram encontradas em '{caminho_completo_arquivo_entrada}': {', '.join(colunas_ausentes_neste_arquivo)}. Elas serão adicionadas como vazias.")

        read_csv_args = {
            'sep': ';',
            'encoding': 'latin1',
            'usecols': colunas_a_realmente_ler,
            'chunksize': chunk_size
        }

        try:
            for chunk in pd.read_csv(caminho_completo_arquivo_entrada, **read_csv_args):
                # Adicionar colunas ausentes no chunk com valores NaN (vazios)
                for col in colunas_ausentes_neste_arquivo:
                    chunk[col] = pd.NA

                if condicao_filtro_funcao:
                    df_filtrado = chunk[chunk.apply(condicao_filtro_funcao, axis=1)]
                else:
                    df_filtrado = chunk

                # Certificar-se de que todas as colunas desejadas estão presentes antes de selecionar
                # e manter a ordem das colunas definidas em colunas_para_salvar
                df_final = pd.DataFrame(columns=colunas_para_salvar)
                for col in colunas_para_salvar:
                    if col in df_filtrado.columns:
                        df_final[col] = df_filtrado[col]
                    else:
                        df_final[col] = pd.NA

                # NOVO PASSO: Converter as colunas especificadas para Int64Dtype
                if colunas_para_int64:
                    for col in colunas_para_int64:
                        if col in df_final.columns:
                            try:
                                df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
                                df_final[col] = df_final[col].astype('Int64')
                            except Exception as e:
                                print(f"Aviso: Não foi possível converter a coluna '{col}' para Int64 no chunk. Erro: {e}")
                        else:
                            print(f"Aviso: Coluna '{col}' não encontrada no DataFrame para conversão para Int64 neste chunk.")

                if primeiro_arquivo:
                    df_final.to_csv(caminho_completo_saida, mode='w', index=False)
                    primeiro_arquivo = False
                else:
                    df_final.to_csv(caminho_completo_saida, mode='a', header=False, index=False)
        except Exception as e:
            print(f"Erro ao ler ou processar o arquivo {caminho_completo_arquivo_entrada} em chunks: {e}")
            continue

    print(f"Processamento concluído. Saída salva em {caminho_completo_saida}")

### Discentes 

In [10]:
#Definição de filtro 
def filtro_ufrj_sigla(linha):
    """
    Verifica se a linha atende aos critérios de filtro:
    - 'SG_ENTIDADE_ENSINO' é 'UFRJ'.

    Args:
        linha (pd.Series): Uma linha do DataFrame.

    Returns:
        bool: True se a linha atende aos critérios, False caso contrário.
    """
    condicao_entidade = linha['SG_ENTIDADE_ENSINO'].strip().upper() == 'UFRJ'

    return condicao_entidade 

In [11]:
#Obtendo lista de arquivos csv com ano de referencia = 2013 ou maior (sem filtragem por regex)
discentes_csvs = selecionar_csvs_capes('download_dir', 'discentes')

Buscando arquivos no diretório: capes_csv_files/discentes


In [12]:
discentes_csvs

['capes_csv_files/discentes/br-capes-colsucup-discentes-2021-2025-03-31.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2013-2021-03-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2022-2025-03-31.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2014-2021-03-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2020-2023-12-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2016-2021-03-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2023-2025-03-31.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2018-2023-12-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2015-2021-03-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2019-2023-12-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2017-2023-12-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2024-2025-12-01.csv']

In [15]:
fundir_lista_csvs(discentes_csvs, 
                colunas_desejadas=['AN_BASE', 'ID_PESSOA', 'CD_PROGRAMA_IES', 'NM_DISCENTE', 'DS_TIPO_NACIONALIDADE_DISCENTE', 
                                    'NM_PAIS_NACIONALIDADE_DISCENTE', 'AN_NASCIMENTO_DISCENTE', 'DS_FAIXA_ETARIA',
                                    'DS_GRAU_ACADEMICO_DISCENTE', 'ST_INGRESSANTE', 'NM_SITUACAO_DISCENTE',
                                    'QT_MES_TITULACAO', 'SG_ENTIDADE_ENSINO'], 
                                    condicao_filtro_funcao=filtro_ufrj_sigla,
                                    diretorio_saida=filtered_dir,
                                    colunas_para_int64=['QT_MES_TITULACAO'],
                                    nome_arquivo_saida='discentes.csv'
                )

Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2021-2025-03-31.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2013-2021-03-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2022-2025-03-31.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2014-2021-03-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2020-2023-12-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2016-2021-03-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2023-2025-03-31.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2018-2023-12-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2015-2021-03-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2019-2023-12-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2017-2023-12-01.csv...
Processando capes_csv

### Docentes

In [16]:
docentes_csvs = selecionar_csvs_capes('download_dir', 'docentes')

Buscando arquivos no diretório: capes_csv_files/docentes


In [17]:
docentes_csvs

['capes_csv_files/docentes/br-capes-colsucup-docente-2023-2025-03-31.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2013-2023-08-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2019-2021-11-10.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2022-2025-03-31.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2015-2023-08-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2020-2021-11-10.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2024-2025-12-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2017-2021-11-10.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2021-2025-03-31.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2016-2023-08-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2014-2023-08-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2018-2021-11-10.csv']

In [18]:
fundir_lista_csvs(docentes_csvs, 
                  colunas_desejadas=['AN_BASE', 'ID_PESSOA', 'CD_PROGRAMA_IES',
                  'NM_DOCENTE', 'AN_NASCIMENTO_DOCENTE', 'DS_FAIXA_ETARIA', 'DS_TIPO_NACIONALIDADE_DOCENTE',
                  'NM_PAIS_NACIONALIDADE_DOCENTE', 'DS_CATEGORIA_DOCENTE', 
                  'DS_TIPO_VINCULO_DOCENTE_IES', 'DS_REGIME_TRABALHO',
                  'CD_CAT_BOLSA_PRODUTIVIDADE', 'NM_GRAU_TITULACAO', 'SG_ENTIDADE_ENSINO'], 
                  condicao_filtro_funcao=filtro_ufrj_sigla,
                  diretorio_saida=filtered_dir,
                  nome_arquivo_saida='docentes.csv')

Processando capes_csv_files/docentes/br-capes-colsucup-docente-2023-2025-03-31.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2013-2023-08-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2019-2021-11-10.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2022-2025-03-31.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2015-2023-08-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2020-2021-11-10.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2024-2025-12-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2017-2021-11-10.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2021-2025-03-31.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2016-2023-08-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2014-2023-08-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup

### Programas

In [19]:
programas_csvs = selecionar_csvs_capes('download_dir', 'programas', padroes_regex=[r"br-capes-colsucup-prog"])

Buscando arquivos no diretório: capes_csv_files/programas


In [20]:
programas_csvs

['capes_csv_files/programas/br-capes-colsucup-prog-2017-2021-11-10.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2013.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2019-2021-11-10.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2022-2025-03-31.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2024-2025-12-01.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2021-2025-03-31.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2023-2025-03-31.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2020-2021-11-10.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2018-2021-11-10.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2015.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2016.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2014.csv']

In [21]:
def filtro_programa(linha):
    """
    Verifica se a linha atende aos critérios de filtro:
    - 'SG_ENTIDADE_ENSINO' é 'UFRJ'.
    - 'SG_ENTIDADE_ENSINO_REDE' contêm 'UFRJ' (caso seja um programa em rede).

    Args:
        linha (pd.Series): Uma linha do DataFrame.

    Returns:
        bool: True se a linha atende aos critérios, False caso contrário.
    """
    condicao_entidade = linha['SG_ENTIDADE_ENSINO'].strip().upper() == 'UFRJ'

    if linha['IN_REDE'] == 'SIM': #Se programa estiver em rede, checa se UFRJ está dentre as IES participantes
        condicao_entidade_rede = 'UFRJ' in linha['SG_ENTIDADE_ENSINO_REDE'].split(';')
    else:
        condicao_entidade_rede = False
    
    return condicao_entidade or condicao_entidade_rede

In [22]:
fundir_lista_csvs(programas_csvs, ['AN_BASE', 'CD_PROGRAMA_IES', 'NM_PROGRAMA_IES', 'NM_GRANDE_AREA_CONHECIMENTO', 'NM_AREA_CONHECIMENTO',
                                    'NM_GRAU_PROGRAMA', 'CD_CONCEITO_PROGRAMA', 'ANO_INICIO_PROGRAMA', 'AN_INICIO_PROGRAMA',
                                    'AN_INICIO_CURSO', 'IN_REDE', 'DS_SITUACAO_PROGRAMA',
                                    'CD_AREA_AVALIACAO', 'NM_AREA_AVALIACAO', 'NM_MODALIDADE_PROGRAMA', 'SG_ENTIDADE_ENSINO',
                                    'SG_ENTIDADE_ENSINO_REDE'
                                   ], 
                                   condicao_filtro_funcao=filtro_programa,
                                   diretorio_saida=filtered_dir,
                                   nome_arquivo_saida='programas.csv'
                                   )

Processando capes_csv_files/programas/br-capes-colsucup-prog-2017-2021-11-10.csv...
Aviso: As seguintes colunas desejadas não foram encontradas em 'capes_csv_files/programas/br-capes-colsucup-prog-2017-2021-11-10.csv': ANO_INICIO_PROGRAMA. Elas serão adicionadas como vazias.
Processando capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2013.csv...
Aviso: As seguintes colunas desejadas não foram encontradas em 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2013.csv': AN_INICIO_PROGRAMA. Elas serão adicionadas como vazias.
Processando capes_csv_files/programas/br-capes-colsucup-prog-2019-2021-11-10.csv...
Aviso: As seguintes colunas desejadas não foram encontradas em 'capes_csv_files/programas/br-capes-colsucup-prog-2019-2021-11-10.csv': ANO_INICIO_PROGRAMA. Elas serão adicionadas como vazias.
Processando capes_csv_files/programas/br-capes-colsucup-prog-2022-2025-03-31.csv...
Aviso: As seguintes colunas desejadas não foram encontradas em 'capes

### Produção (artigos de periódicos por autor)

In [23]:
producao_csvs = selecionar_csvs_capes('download_dir', 'producao', padroes_regex=[r"br-capes-colsucup-prod-autor-.*bibliografica-artpe"])

Buscando arquivos no diretório: capes_csv_files/producao


In [24]:
producao_csvs

['capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2022.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2013a2016-2017-03-01-bibliografica-artpe.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2019.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2023.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2021.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-12-01-bibliografica-artpe-2024.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2018.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2020.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2017.csv',
 'capes_csv_files/producao/br-cap

In [25]:
def filtro_producao(linha):
    """
    Verifica se a linha atende aos critérios de filtro:
    - 'SG_ENTIDADE_ENSINO' é 'UFRJ'.
    - 'ID_PESSOA_DOCENTE' não é nulo.
    - 'ID_PESSOA_DISCENTE' não é nulo.
    - 'NM_NIVEL_DISCENTE' está em algum nível que corresponda a alunos de pós.

    Args:
        linha (pd.Series): Uma linha do DataFrame.

    Returns:
        bool: True se a linha atende aos critérios, False caso contrário.
    """
    condicao_entidade = linha['SG_ENTIDADE_ENSINO'].strip().upper() == 'UFRJ'
    condicao_discente_nao_nulo = pd.notna(linha['ID_PESSOA_DISCENTE'])
    condicao_docente_nao_nulo = pd.notna(linha['ID_PESSOA_DOCENTE'])

    condicao_nivel_discente = True #Se a linha não estiver se referindo a discente, isso deve ser verdadeiro para não excluir o registro

    if condicao_discente_nao_nulo:
        nm_nivel_discente = str(linha.get('NM_NIVEL_DISCENTE', '')).strip().upper()
        condicao_nivel_discente = (nm_nivel_discente in ['MESTRADO', 'DOUTORADO', 'MESTRADO PROFISSIONAL', 'DOUTORADO PROFISSIONAL']) #Só inclui alunos de pós 
    
    return condicao_entidade and (condicao_discente_nao_nulo or condicao_docente_nao_nulo) and condicao_nivel_discente

In [26]:
fundir_lista_csvs(producao_csvs, 
                colunas_desejadas=['AN_BASE', 'CD_PROGRAMA_IES', 'NM_PROGRAMA_IES', 'ID_ADD_PRODUCAO_INTELECTUAL', 
                                   'ID_PESSOA_DOCENTE', 'ID_PESSOA_DISCENTE', 'TP_AUTOR', 
                                   'NM_TP_CATEGORIA_DOCENTE', 'NM_NIVEL_DISCENTE', 'SG_ENTIDADE_ENSINO'], 
                condicao_filtro_funcao=filtro_producao, 
                diretorio_saida=filtered_dir,
                nome_arquivo_saida='producao.csv',
                colunas_para_int64= ['ID_PESSOA_DOCENTE', 'ID_PESSOA_DISCENTE'] ,
               )

Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2022.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2013a2016-2017-03-01-bibliografica-artpe.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2019.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2023.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2021.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-12-01-bibliografica-artpe-2024.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2018.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2020.csv...
Processando capes_csv_files/producao/br-capes-colsucu

## Processamento/limpeza dos dados da ufrj

- Discentes:
    - Geração da coluna de gênero/remoção da coluna de nome
    - Converter ST_INGRESSANTE para booleano

- Docentes: 
    - Geração da coluna de gênero/remoção da coluna de nome
    - Limpeza campo CD_CAT_BOLSA_PRODUTIVIDADE (remover os NA e SR)

- Pessoas:
    - Pegar os campos de docentes e discentes que não mudam ao longo dos anos

- Programas:
    - Campo IN_REDE convertido para booleano
    - Substituir conceito 'A' ('Ausente') por 0
    - Fundir colunas ANO_INICIO_PROGRAMA (2013-2016) e AN_INICIO_PROGRAMA (2017 em diante)
    - Gerar a tabela 'programa' (informações que não mudam ao longo dos anos)
    - Gerar a tabela 'ano_programa', contendo apenas as informações sobre os programas que podem mudar ao longo dos anos
    - Gerar a tabela 'cursos', com a relação entre CD_PROGRAMA_IES, NM_GRAU_PROGRAMA e ANO_INICIO_CURSO
        - Separar os valores separados por barra em NM_GRAU_PROGRAMA (e.g. MESTRADO/DOUTORADO) e ANO_INICIO_CURSO (e.g. 1981/2001)
        - Adicionar outras linhas normalmente

- Produção:
    - Unir as colunas ID_PESSOA_DOCENTE e ID_PESSOA_DISCENTE em uma só

- Todas:
    - Remover as colunas usadas para a filtragem dos dados
    - Quando estiver presente, substituir o ID_PESSOA por um hash (ID_PESSOA + salt): Salvar a relação ID_PESSOA e salt em uma db sqlite separada

In [27]:
#Importando dicionário de gêneros
with open('aux/dicionario_generos.json', 'r') as f: 
    dicionario_generos = json.load(f)

dicionario_generos

{'AALINE': 'F',
 'AILINE': 'F',
 'ALEINE': 'F',
 'ALIINE': 'F',
 'ALINE': 'F',
 'ALINER': 'F',
 'ALINHE': 'F',
 'ALINNE': 'F',
 'ALYNE': 'F',
 'ALYNNE': 'F',
 'AYLINE': 'F',
 'EALINE': 'F',
 'ELEINE': 'F',
 'ELINE': 'F',
 'ELINER': 'F',
 'ELINNE': 'F',
 'ELYNE': 'F',
 'EULINE': 'F',
 'HALINE': 'F',
 'HALYNE': 'F',
 'HELEINE': 'F',
 'HELINE': 'F',
 'HELYNE': 'F',
 'IALINE': 'F',
 'ILEINE': 'F',
 'ILINE': 'F',
 'LEINE': 'F',
 'LEINER': 'F',
 'LEYNE': 'F',
 'LINE': 'F',
 'LINER': 'F',
 'LUEINE': 'F',
 'LUINE': 'F',
 'LUYNE': 'F',
 'LYNE': 'F',
 'LYNNE': 'F',
 'OLINE': 'F',
 'UELINE': 'F',
 'AARAO': 'M',
 'ARAAO': 'M',
 'ARAO': 'M',
 'AARON': 'M',
 'AHARON': 'M',
 'AROM': 'M',
 'ARON': 'M',
 'ARYON': 'M',
 'HARON': 'M',
 'ABA': 'F',
 'ADA': 'F',
 'ADAH': 'F',
 'ADAR': 'F',
 'ADHA': 'F',
 'HADA': 'F',
 'ABADE': 'M',
 'ABADI': 'M',
 'ABADIR': 'M',
 'ABADIA': 'F',
 'ABADIAS': 'M',
 'ABADIO': 'M',
 'ABAETE': 'F',
 'ABETE': 'F',
 'ADETE': 'F',
 'ABD': 'M',
 'ABDA': 'F',
 'ADDA': 'F',
 'ABDAEL':

In [28]:
# Define a função auxiliar que será aplicada a cada nome completo
def genero_baseado_no_primeiro_nome(nome_completo: str,
                    dicionario_generos: dict) -> str:
    
    # Garante que é uma string, útil para lidar com NaNs ou outros tipos
    nome_completo_str = str(nome_completo) 
    
    primeiro_nome = nome_completo_str.split(' ')[0].strip().upper()
    
    # Usa .get() para retornar 'D' (Desconhecido) se o nome não for encontrado
    return dicionario_generos.get(primeiro_nome, 'D')


def adicionar_coluna_genero(df,
                                 coluna_nome_completo: str, 
                                 dicionario_generos: dict,
                                 coluna_genero: str = 'TP_SEXO',
                       ):
    """
    Extrai o primeiro nome de uma coluna, consulta um dicionário de gêneros baseados em primeiros nomes
    e retorna os gêneros correspondentes em uma nova coluna, usando df.apply().

    Args:
        df (pd.DataFrame): O DataFrame de entrada.
        coluna_nome_completo (str): O nome da coluna no DataFrame que contém os nomes completos.
        dicionario_generos (dict): Um dicionário onde as chaves são os primeiros nomes (em maiúsculas)
                                   e os valores são os gêneros ('F', 'M', 'Desconhecido', etc.).
        coluna_genero (str): Nome da nova coluna com o genero inferido com base no primeiro nome. O padrão é 'GN_PESSOA'.

    Returns:
        pd.DataFrame: O DataFrame original com uma nova coluna .
    """
    df[coluna_genero] = df[coluna_nome_completo].apply(genero_baseado_no_primeiro_nome, 
                                                       dicionario_generos=dicionario_generos)
    
    return df

In [29]:
#Criando diretório de saída dos arquivos processados
try:
    # Cria o diretório
    # 'exist_ok=True' é crucial: se o diretório já existir, ele não levantará um erro (FileExistsError)
    os.makedirs(processed_dir, exist_ok=True)
    print(f"Diretório '{processed_dir}' criado com sucesso ou já existente.")
except OSError as e:
    # Trata outros possíveis erros do sistema operacional (permissões, nomes inválidos, etc.)
    print(f"Erro ao criar o diretório '{processed_dir}': {e}")

Diretório 'sucupira_painel' criado com sucesso ou já existente.


### Substituição de IDs por hashes (anonimização dos dados)

A idéia aqui é:

- Pegar todos os ids únicos dos dados da CAPES
- Gerar um salt aleatório para cada um
- Fazer o hashing do id_original + salt
- Salvar a correspondência entre id_original, salt e hash em uma db sqlite encriptada 
    - Essa db servirá como armazenamento persistente para cenários (improváveis) em que precisemos retornar a algum id_original a partir do hash
- Gerar um dicionário com a relação entre id_original e hash
    - Com base no dicionário, substituir o id_original pelo hash nas tabelas para anonimizar os dados

In [30]:
# --- Configuração ---
# Carrega as variáveis de ambiente do arquivo .env primeiro.
# Usar override=True garante que as alterações no .env sejam lidas sem reiniciar o kernel.
load_dotenv(override=True) 

#Salvando em variáveis as informações do dotenv sobre a database que irá guardar os id_pessoa e seus respectivos salts e hashes:
DATABASE_FILEPATH = os.getenv('DATABASE_FILEPATH') #Caminho para a database
DATABASE_PASSWORD = os.getenv('DATABASE_PASSWORD') #Senha da database


# --- Mensagens de Depuração da Configuração --
print(f"DEBUG: Caminho do arquivo do banco de dados: '{DATABASE_FILEPATH}'")
#print(f"DEBUG: DATABASE_PASSWORD carregado (primeiros caracteres): '{DATABASE_PASSWORD[:4]}...'") # Evita imprimir a senha completa
print(f"DEBUG: Número de núcleos da CPU detectados: {os.cpu_count()}")


# --- Função de Configuração do Banco de Dados ---
def setup_secure_database():
    """
    Configura e criptografa o banco de dados SQLite, se ele não existir.
    
    A tabela é genérica e pode armazenar hashes para qualquer tipo de ID.
    """
    print(f"\n--- Configurando Banco de Dados Seguro ---")
    print(f"DEBUG: Tentando conectar a: {DATABASE_FILEPATH}")
    try:
        # Usa a declaração 'with' para tratamento automático da conexão
        with sqlite.connect(DATABASE_FILEPATH) as conn: 
            conn.execute(f"PRAGMA key = '{DATABASE_PASSWORD}';")
            conn.execute("PRAGMA cipher_migrate;") # Garante que a criptografia mais recente seja aplicada ao banco sqlite
            
            cursor = conn.cursor()
            print("DEBUG: Tentando CRIAR TABELA SE NÃO EXISTIR 'identificadores_privados'.")
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS identificadores_privados (
                    id_original TEXT PRIMARY KEY,
                    salt BLOB NOT NULL UNIQUE,
                    hash_public_fixo TEXT NOT NULL UNIQUE,
                    data_geracao TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    data_ultima_atualizacao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
            ''')
            conn.commit() # Confirma explicitamente as alterações
            print(f"Banco de dados '{DATABASE_FILEPATH}' configurado e criptografado com sucesso.")
            
    except sqlite.Error as e:
        print(f"ERRO na configuração do banco de dados seguro: {e}")
        
# --- Funções Auxiliares para Hashing ---
def generate_salt(length=16):
    """Gera um salt aleatório."""
    return os.urandom(length)

def generate_hashed_id(id_and_salt_tuple):
    """
    Gera um hash SHA256 usando o ID original e um salt,
    recebendo-os como uma tupla para uso em paralelismo.
    """
    original_id, salt = id_and_salt_tuple # Desempacota a tupla
    
    id_bytes = str(original_id).encode('utf-8')
    combined = id_bytes + salt
    new_hash_public = hashlib.sha256(combined).hexdigest() #Faz o hash e o converte para string
    
    # Retorna a tupla completa (id_original, salt, hash)
    # que é o formato esperado para a inserção no DB.
    return (original_id, salt, new_hash_public)


# --- Lógica Principal (Paralelismo) ---
def get_or_create_hashed_ids_secure_parallel(original_ids_list, batch_size=5000, max_workers=os.cpu_count()): # batch_size é útil para inserção em lote
    results = {}
    ids_and_salts_for_parallel_hashing = [] 

    print(f"\n--- Iniciando Processamento Paralelo para {len(original_ids_list):,} IDs ---")

    # Passo 1: Pré-verificação sequencial de IDs existentes no banco de dados (fase de leitura)
    try:
        with sqlite.connect(DATABASE_FILEPATH) as conn:
            conn.execute(f"PRAGMA key = '{DATABASE_PASSWORD}';")
            cursor = conn.cursor()

            if original_ids_list:
                # Divide a lista para verificar IDs existentes para evitar cláusulas IN muito grandes
                chunk_size_check = 1000 # Ajuste conforme necessário
                for i in range(0, len(original_ids_list), chunk_size_check):
                    chunk = original_ids_list[i:i + chunk_size_check]
                    placeholders = ','.join('?' for _ in chunk)
                    cursor.execute(f'SELECT id_original, hash_public_fixo FROM identificadores_privados WHERE id_original IN ({placeholders})', chunk)
                    for row in cursor.fetchall():
                        original_id, hash_public_fixo = row
                        results[original_id] = hash_public_fixo
                
    except sqlite.Error as e:
        print(f"ERROR durante a leitura inicial do DB para IDs: {e}")
        
    for original_id in original_ids_list:
        if original_id not in results:
            ids_and_salts_for_parallel_hashing.append((original_id, generate_salt()))

    print(f"DEBUG: {len(results):,} IDs encontrados no DB. {len(ids_and_salts_for_parallel_hashing):,} IDs precisam de novos hashes (processamento paralelo).")

    # Passo 2: Paraleliza o hashing para novos IDs
    new_hashes_data = []
    if ids_and_salts_for_parallel_hashing:
        print(f"DEBUG: Iniciando hashing paralelo para {len(ids_and_salts_for_parallel_hashing):,} IDs...")
        parallel_start_time = time.time()

        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            # map submete tarefas e coleta resultados na ordem
            new_hashes_data = list(executor.map(generate_hashed_id, ids_and_salts_for_parallel_hashing))

        parallel_end_time = time.time()
        print(f"DEBUG: Hashing paralelo concluído! Levou {parallel_end_time - parallel_start_time:.2f} segundos.")
    else:
        print("DEBUG: Nenhum ID novo para fazer hashing em paralelo.")


    # Passo 3: Insere novos hashes no banco de dados sequencialmente (escrita em lote)
    if new_hashes_data:
        print(f"DEBUG: Iniciando inserção em lote sequencial para {len(new_hashes_data):,} novos hashes...")
        insert_start_time = time.time()

        try:
            with sqlite.connect(DATABASE_FILEPATH) as conn:
                conn.execute(f"PRAGMA key = '{DATABASE_PASSWORD}';")
                cursor = conn.cursor()

                for i in range(0, len(new_hashes_data), batch_size):
                    chunk_to_insert = new_hashes_data[i:i + batch_size]
                    cursor.executemany('''
                        INSERT OR IGNORE INTO identificadores_privados (id_original, salt, hash_public_fixo)
                        VALUES (?, ?, ?)
                    ''', chunk_to_insert)
                    conn.commit()

            print(f"DEBUG: Inserção em lote concluída! Levou {time.time() - insert_start_time:.2f} segundos.")

            # Re-leitura simples para garantir que todos os IDs estejam no dicionário `results`
            # Este passo é crucial para garantir que os hashes de IDs que já existiam sejam recuperados
            print("DEBUG: Fazendo uma leitura final para garantir que todos os hashes estejam no resultado.")

            # Obter a lista completa de IDs originais para a leitura final
            all_original_ids = [item[0] for item in new_hashes_data]

            if all_original_ids:
                chunk_size_check_final = 1000
                with sqlite.connect(DATABASE_FILEPATH) as conn:
                    conn.execute(f"PRAGMA key = '{DATABASE_PASSWORD}';")
                    cursor = conn.cursor()
                    for i in range(0, len(all_original_ids), chunk_size_check_final):
                        chunk = all_original_ids[i:i + chunk_size_check_final]
                        placeholders = ','.join('?' for _ in chunk)
                        cursor.execute(f'SELECT id_original, hash_public_fixo FROM identificadores_privados WHERE id_original IN ({placeholders})', chunk)
                        for row in cursor.fetchall():
                            original_id, hash_public_fixo = row
                            results[original_id] = hash_public_fixo

        except sqlite.Error as e:
            print(f"ERROR durante a inserção ou leitura final: {e}")
            
    print(f"--- Processamento de IDs Concluído ---")
    return results

DEBUG: Caminho do arquivo do banco de dados: 'id_hash_painel_gid.db'
DEBUG: Número de núcleos da CPU detectados: 12


In [31]:
#Configurando a db que irá receber os dados sobre o hashing
setup_secure_database()


--- Configurando Banco de Dados Seguro ---
DEBUG: Tentando conectar a: id_hash_painel_gid.db
DEBUG: Tentando CRIAR TABELA SE NÃO EXISTIR 'identificadores_privados'.
Banco de dados 'id_hash_painel_gid.db' configurado e criptografado com sucesso.


In [32]:
#Selecionando e renomeando as colunas de ids a serem hasheadas
id_pessoa_docente = pd.read_csv(f'{filtered_dir}/discentes.csv', usecols=['ID_PESSOA']).rename(columns = {'ID_PESSOA': 'ID'})
id_pessoa_discente = pd.read_csv(f'{filtered_dir}/docentes.csv', usecols=['ID_PESSOA']).rename(columns = {'ID_PESSOA': 'ID'})
id_producao = pd.read_csv(f'{filtered_dir}/producao.csv', usecols=['ID_ADD_PRODUCAO_INTELECTUAL']).rename(columns = {'ID_ADD_PRODUCAO_INTELECTUAL': 'ID'})
id_programa = pd.read_csv(f'{filtered_dir}/programas.csv', usecols=['CD_PROGRAMA_IES']).rename(columns = {'CD_PROGRAMA_IES': 'ID'})

In [33]:
#Concatenando e removendo duplicatas
ids_originais = pd.concat([id_pessoa_discente, id_pessoa_docente, id_producao, id_programa], ignore_index=True)['ID'].drop_duplicates()
ids_originais.is_unique #Checando se realmente só temos valores unicos

True

In [34]:
#Convertendo os dados em uma lista de strings (banco de dados espera que ID seja string)
lista_ids_originais_str = ids_originais.astype(str).tolist()

In [35]:
#Criando um dicionario que vai conter a relação entre os ids originais e seus hash correspondente
hash_lookup = get_or_create_hashed_ids_secure_parallel(lista_ids_originais_str)


--- Iniciando Processamento Paralelo para 154,450 IDs ---
DEBUG: 154,450 IDs encontrados no DB. 0 IDs precisam de novos hashes (processamento paralelo).
DEBUG: Nenhum ID novo para fazer hashing em paralelo.
--- Processamento de IDs Concluído ---


In [36]:
# Criando uma funcao que recebe uma série de ids devolve uma série modificada com os valores de hash
def converter_ids_para_hashes(serie_de_ids: pd.Series) -> pd.Series:
    """
    Converte uma Série de IDs para uma Série de hashes usando um dicionário de lookup.

    Args:
        serie_de_ids (pd.Series): Uma coluna (Série) do pandas contendo os IDs.

    Returns:
        pd.Series: Uma nova Série com os valores de hash correspondentes.
                   Retorna NaN para IDs não encontrados no dicionário.
    """
    # O método .astype(str) converte todos os ids para strings (que é como a db armazena os ids).
    # O método .map() faz todo o trabalho de procurar cada ID no dicionário.
    serie_de_hashes = serie_de_ids.astype(str).map(hash_lookup)
    return serie_de_hashes

### Discentes

In [37]:
#Importando dados dos discentes
discentes = pd.read_csv(f'{filtered_dir}/discentes.csv')
discentes.head()

,AN_BASE,ID_PESSOA,CD_PROGRAMA_IES,NM_DISCENTE,DS_TIPO_NACIONALIDADE_DISCENTE,NM_PAIS_NACIONALIDADE_DISCENTE,AN_NASCIMENTO_DISCENTE,DS_FAIXA_ETARIA,DS_GRAU_ACADEMICO_DISCENTE,ST_INGRESSANTE,NM_SITUACAO_DISCENTE,QT_MES_TITULACAO,SG_ENTIDADE_ENSINO
0,2021,3353229,31001017100P2,BRENDO ARAUJO GOMES,BRASILEIRO,BRASIL,1994,25 A 29 ANOS,DOUTORADO,SIM,MATRICULADO,0,UFRJ
1,2021,26505,31001017033P3,CLAUDIA BENITEZ LOGELO,BRASILEIRO,BRASIL,1973,45 A 49 ANOS,DOUTORADO,NÃO,MATRICULADO,0,UFRJ
2,2021,1216819,31001017172P3,FRANCIANE PIMENTEL MELO,BRASILEIRO,BRASIL,1974,45 A 49 ANOS,MESTRADO,NÃO,MATRICULADO,0,UFRJ
3,2021,794525,31001017020P9,GABRIELA MONTEZ HOLANDA DA SILVA,BRASILEIRO,BRASIL,1990,30 A 34 ANOS,DOUTORADO,NÃO,MATRICULADO,0,UFRJ
4,2021,4485640,31001017134P4,DANIELLE BRODA DE VASCONCELLOS,BRASILEIRO,BRASIL,1991,30 A 34 ANOS,MESTRADO PROFISSIONAL,SIM,MATRICULADO,0,UFRJ


In [38]:
#Gerando colunas com os hashes baseados nos ids originais
discentes['ID_PESSOA_HASH'] = converter_ids_para_hashes(discentes['ID_PESSOA'])
discentes['ID_PROGRAMA_HASH'] = converter_ids_para_hashes(discentes['CD_PROGRAMA_IES'])
discentes[['ID_PESSOA', 'ID_PESSOA_HASH', 'CD_PROGRAMA_IES', 'ID_PROGRAMA_HASH']]

,ID_PESSOA,ID_PESSOA_HASH,CD_PROGRAMA_IES,ID_PROGRAMA_HASH
0,3353229,3e904591534275abcc843da1d0a3614e1a7b6d302d236f...,31001017100P2,66751c1feedc37dc12807304bbd673e452000cab1b1755...
1,26505,bb5f299988d99383b4a6e30e92924031638eb4fe3e6b98...,31001017033P3,7b75cd7fbdac4855cb781fe648da4adb61620db94c996e...
2,1216819,7c408169ed58a44e99d6ecfae493feaa0c4deae7ba3e22...,31001017172P3,06e938eaef4be265435f15838c81ba3b2721bbd908dceb...
3,794525,865ea08e85fb688cf728650403234496c664815fa1d7e8...,31001017020P9,2da72851f4776ed03173310560e21621a1e1e0c3626908...
4,4485640,b1048726933afef278ba1ee518e0647781d52241cfb241...,31001017134P4,99ea49d14288358fd2e7eb0305d4476a17efb6ecda6227...
...,...,...,...,...
174316,4320878,8184223ad852aa9e9e2ef46ee58588b50a9b895ea61867...,31001017111P4,6c5c3c7e488b0614df1789a1ebe21262df43cd295ef5e0...
174317,4809377,585688ecb04e3372d981dcfa460f52aee4cd89e02b1974...,31001017132P1,ab073f6b50b70dac250c0d722fd922ce718a5039f89102...
174318,1891217,f9972e72a5efcc8a5361948f6682aa5fcb6580eb0aac32...,31001017028P0,f802b104594ad37dd34281e560156b52d1a7b8eb096099...
174319,3889748,677df6ab1a736fb27d59ed4c8e14ffa3f989ca84231760...,31001017038P5,27848dd22495a7b2b6104f5ab1d6d99c47b305f4f33609...


In [39]:
#Gerando a coluna de sexo com base no primeiro nome
discentes = adicionar_coluna_genero(discentes,
                        coluna_nome_completo='NM_DISCENTE',
                        dicionario_generos=dicionario_generos)
discentes[['NM_DISCENTE', 'TP_SEXO']].head()

,NM_DISCENTE,TP_SEXO
0,BRENDO ARAUJO GOMES,M
1,CLAUDIA BENITEZ LOGELO,F
2,FRANCIANE PIMENTEL MELO,F
3,GABRIELA MONTEZ HOLANDA DA SILVA,F
4,DANIELLE BRODA DE VASCONCELLOS,F


In [40]:
#Convertendo coluna ST_INGRESSANTE para booleano
mapeamento_booleano = {'SIM': True, 'NÃO': False}
discentes['ST_INGRESSANTE'] = discentes['ST_INGRESSANTE'].map(mapeamento_booleano)

In [41]:
#Campo NM_SITUACAO_DISCENTE tem um erro de preenchimento específico onde falta a cedilha em 'MUDANCA DE NÍVEL SEM DEFESA'
#Vamos corrigir isso aqui:
discentes['NM_SITUACAO_DISCENTE'] = discentes['NM_SITUACAO_DISCENTE'].replace('MUDANCA DE NÍVEL SEM DEFESA', 'MUDANÇA DE NÍVEL SEM DEFESA')
discentes['NM_SITUACAO_DISCENTE'].value_counts()


NM_SITUACAO_DISCENTE
MATRICULADO                    136057
TITULADO                        31834
DESLIGADO                        3480
ABANDONOU                        2859
MUDANÇA DE NÍVEL SEM DEFESA        91
Name: count, dtype: int64

In [42]:
#Mantendo apenas as colunas necessárias (que mudam ao longo do tempo)
discentes_final_cols = ['AN_BASE', 'ID_PESSOA_HASH', 'DS_FAIXA_ETARIA', 'ID_PROGRAMA_HASH', 'DS_GRAU_ACADEMICO_DISCENTE', 'ST_INGRESSANTE', 'NM_SITUACAO_DISCENTE', 'QT_MES_TITULACAO']
discentes_final = discentes[discentes_final_cols]

In [43]:
#Visualizando dataframe final
discentes_final.head()

,AN_BASE,ID_PESSOA_HASH,DS_FAIXA_ETARIA,ID_PROGRAMA_HASH,DS_GRAU_ACADEMICO_DISCENTE,ST_INGRESSANTE,NM_SITUACAO_DISCENTE,QT_MES_TITULACAO
0,2021,3e904591534275abcc843da1d0a3614e1a7b6d302d236f...,25 A 29 ANOS,66751c1feedc37dc12807304bbd673e452000cab1b1755...,DOUTORADO,True,MATRICULADO,0
1,2021,bb5f299988d99383b4a6e30e92924031638eb4fe3e6b98...,45 A 49 ANOS,7b75cd7fbdac4855cb781fe648da4adb61620db94c996e...,DOUTORADO,False,MATRICULADO,0
2,2021,7c408169ed58a44e99d6ecfae493feaa0c4deae7ba3e22...,45 A 49 ANOS,06e938eaef4be265435f15838c81ba3b2721bbd908dceb...,MESTRADO,False,MATRICULADO,0
3,2021,865ea08e85fb688cf728650403234496c664815fa1d7e8...,30 A 34 ANOS,2da72851f4776ed03173310560e21621a1e1e0c3626908...,DOUTORADO,False,MATRICULADO,0
4,2021,b1048726933afef278ba1ee518e0647781d52241cfb241...,30 A 34 ANOS,99ea49d14288358fd2e7eb0305d4476a17efb6ecda6227...,MESTRADO PROFISSIONAL,True,MATRICULADO,0


In [44]:
#Salvando dataframe final
discentes_final.to_csv(f'{processed_dir}/discentes.csv', index=False)

### Docentes

In [45]:
#Importando df filtrada
docentes = pd.read_csv(f'{filtered_dir}/docentes.csv')
docentes.head()

,AN_BASE,ID_PESSOA,CD_PROGRAMA_IES,NM_DOCENTE,AN_NASCIMENTO_DOCENTE,DS_FAIXA_ETARIA,DS_TIPO_NACIONALIDADE_DOCENTE,NM_PAIS_NACIONALIDADE_DOCENTE,DS_CATEGORIA_DOCENTE,DS_TIPO_VINCULO_DOCENTE_IES,DS_REGIME_TRABALHO,CD_CAT_BOLSA_PRODUTIVIDADE,NM_GRAU_TITULACAO,SG_ENTIDADE_ENSINO
0,2023,91992,31001017003P7,MARAL MOSTAFAZADEHFARD,1983,40 A 44 ANOS,ESTRANGEIRO,IRÃ,COLABORADOR,SERVIDOR PÚBLICO,INTEGRAL,NaN,DOUTORADO,UFRJ
1,2023,513065,31001017003P7,KATRIN GRIT GELFERT,1973,50 A 54 ANOS,ESTRANGEIRO,ALEMANHA,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,1C,DOUTORADO,UFRJ
2,2023,173737,31001017003P7,SERGIO AUGUSTO ROMANA IBARRA,1984,35 A 39 ANOS,BRASILEIRO,BRASIL,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,NaN,DOUTORADO,UFRJ
3,2023,1129065,31001017003P7,ISAIA NISOLI,1982,40 A 44 ANOS,ESTRANGEIRO,ITÁLIA,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,NaN,DOUTORADO,UFRJ
4,2023,939007,31001017003P7,SEYED HAMID HASSANZADEH HAFSHEJANI,1982,40 A 44 ANOS,ESTRANGEIRO,IRÃ,PERMANENTE,SERVIDOR PÚBLICO,DEDICAÇÃO EXCLUSIVA,NaN,DOUTORADO,UFRJ


In [46]:
#Gerando colunas com os hashes baseados nos ids originais
docentes['ID_PESSOA_HASH'] = converter_ids_para_hashes(docentes['ID_PESSOA'])
docentes['ID_PROGRAMA_HASH'] = converter_ids_para_hashes(docentes['CD_PROGRAMA_IES'])
docentes[['ID_PESSOA', 'ID_PESSOA_HASH', 'CD_PROGRAMA_IES', 'ID_PROGRAMA_HASH']]

,ID_PESSOA,ID_PESSOA_HASH,CD_PROGRAMA_IES,ID_PROGRAMA_HASH
0,91992,7320966aa0e45a8216d20fdfd2873a490e8e282e3da127...,31001017003P7,2842763eea8260182b0be4e10db0271a4b37621331b571...
1,513065,74a1f9ece1d24d8cbd555b86c25f7190650d326d0a0b00...,31001017003P7,2842763eea8260182b0be4e10db0271a4b37621331b571...
2,173737,0ed6dc6dd0e3f0ba91d13197fb6feae1f5c1f2d1a9e3e3...,31001017003P7,2842763eea8260182b0be4e10db0271a4b37621331b571...
3,1129065,0765aa0bd35fc68ec47fc4aca5e853cd0291c1785f9d9b...,31001017003P7,2842763eea8260182b0be4e10db0271a4b37621331b571...
4,939007,1109c9204a042381fb6c132c26268cc196761f3f14e518...,31001017003P7,2842763eea8260182b0be4e10db0271a4b37621331b571...
...,...,...,...,...
42360,976903,e6a847b74e40d7a91afd475c69215fd5be5577d0fe3e89...,31001017066P9,8563d7b42b41188f6ab64dde9a1fb17124cb8c23477305...
42361,534954,455d3e34df1bcd706dc8da5372e433ee35f4227e3f6977...,31001017128P4,b467773fbcc10d8043a6af8754993ac5bddc5b71afd067...
42362,16509,d464d11421ef42f1617b719cc8fe2945f07d034eea29ac...,31001017028P0,f802b104594ad37dd34281e560156b52d1a7b8eb096099...
42363,709792,6cb8b1a40f58aed41b50df80b7519b1f088dcff84671f0...,31001017138P0,867609cf3145d4b8a8e4dc4a88679b5402afb79bd93698...


In [47]:
#Gerando a coluna de gênero com base no primeiro nome
docentes = adicionar_coluna_genero(docentes,
                        coluna_nome_completo='NM_DOCENTE',
                        dicionario_generos=dicionario_generos)
docentes[['NM_DOCENTE', 'TP_SEXO']].head()

,NM_DOCENTE,TP_SEXO
0,MARAL MOSTAFAZADEHFARD,D
1,KATRIN GRIT GELFERT,F
2,SERGIO AUGUSTO ROMANA IBARRA,M
3,ISAIA NISOLI,M
4,SEYED HAMID HASSANZADEH HAFSHEJANI,D


In [48]:
#limpeza do campo CD_CAT_PRODUTIVIDADE
docentes['CD_CAT_BOLSA_PRODUTIVIDADE'].unique()

array([nan, '1C', '1D', '1B', '1A', '2', 'SR'], dtype=object)

In [49]:
docentes['CD_CAT_BOLSA_PRODUTIVIDADE'] = docentes['CD_CAT_BOLSA_PRODUTIVIDADE'].replace([np.nan, 'SR'], pd.NA)
docentes['CD_CAT_BOLSA_PRODUTIVIDADE'].unique()

array([<NA>, '1C', '1D', '1B', '1A', '2'], dtype=object)

In [50]:
#Checando se só temos docentes permanentes
docentes['DS_CATEGORIA_DOCENTE'].unique()

array(['COLABORADOR', 'PERMANENTE', 'VISITANTE'], dtype=object)

In [51]:
#Mantendo apenas colunas necessárias (que mudam ao longo do tempo)
docentes_final_cols = ['AN_BASE', 'ID_PESSOA_HASH', 'DS_FAIXA_ETARIA', 'ID_PROGRAMA_HASH', 'DS_CATEGORIA_DOCENTE', 'DS_TIPO_VINCULO_DOCENTE_IES', 'DS_REGIME_TRABALHO', 'CD_CAT_BOLSA_PRODUTIVIDADE', 'NM_GRAU_TITULACAO']
docentes_final = docentes[docentes_final_cols]

In [52]:
#Visualizando resultado final
docentes_final.head()

,AN_BASE,ID_PESSOA_HASH,DS_FAIXA_ETARIA,ID_PROGRAMA_HASH,DS_CATEGORIA_DOCENTE,DS_TIPO_VINCULO_DOCENTE_IES,DS_REGIME_TRABALHO,CD_CAT_BOLSA_PRODUTIVIDADE,NM_GRAU_TITULACAO
0,2023,7320966aa0e45a8216d20fdfd2873a490e8e282e3da127...,40 A 44 ANOS,2842763eea8260182b0be4e10db0271a4b37621331b571...,COLABORADOR,SERVIDOR PÚBLICO,INTEGRAL,<NA>,DOUTORADO
1,2023,74a1f9ece1d24d8cbd555b86c25f7190650d326d0a0b00...,50 A 54 ANOS,2842763eea8260182b0be4e10db0271a4b37621331b571...,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,1C,DOUTORADO
2,2023,0ed6dc6dd0e3f0ba91d13197fb6feae1f5c1f2d1a9e3e3...,35 A 39 ANOS,2842763eea8260182b0be4e10db0271a4b37621331b571...,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,<NA>,DOUTORADO
3,2023,0765aa0bd35fc68ec47fc4aca5e853cd0291c1785f9d9b...,40 A 44 ANOS,2842763eea8260182b0be4e10db0271a4b37621331b571...,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,<NA>,DOUTORADO
4,2023,1109c9204a042381fb6c132c26268cc196761f3f14e518...,40 A 44 ANOS,2842763eea8260182b0be4e10db0271a4b37621331b571...,PERMANENTE,SERVIDOR PÚBLICO,DEDICAÇÃO EXCLUSIVA,<NA>,DOUTORADO


In [53]:
#Salvando dataframe final
docentes_final.to_csv(f'{processed_dir}/docentes.csv', index=False)

### Pessoas

In [54]:
def padronizar_nomes_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renomeia colunas que terminam com '_DOCENTE' ou '_DISCENTE' para nomes padronizados.
    Ex: 'AN_NASCIMENTO_DOCENTE' -> 'AN_NASCIMENTO'
        'NM_PAIS_NACIONALIDADE_DISCENTE' -> 'NM_PAIS_NACIONALIDADE'

    Args:
        df (pd.DataFrame): O DataFrame a ser renomeado.

    Returns:
        pd.DataFrame: O DataFrame com as colunas renomeadas.
    """
    novo_mapeamento = {}
    for coluna in df.columns:
        if coluna.endswith('_DOCENTE'):
            novo_mapeamento[coluna] = coluna.replace('_DOCENTE', '')
        elif coluna.endswith('_DISCENTE'):
            novo_mapeamento[coluna] = coluna.replace('_DISCENTE', '')
        else:
            novo_mapeamento[coluna] = coluna # Mantém colunas sem sufixo inalteradas
    
    return df.rename(columns=novo_mapeamento)

In [55]:
#Padronizando nomes de colunas entre docentes e discentes e extraindo colunas da tabela 'pessoas'
pessoas_cols = ['AN_BASE', 'ID_PESSOA_HASH', 'NM', 'AN_NASCIMENTO', 'NM_PAIS_NACIONALIDADE', 'DS_TIPO_NACIONALIDADE', 'TP_SEXO']
pessoas_docentes = padronizar_nomes_colunas(docentes)[pessoas_cols]
pessoas_discentes = padronizar_nomes_colunas(discentes)[pessoas_cols]

In [56]:
pessoas = pd.concat([pessoas_discentes, pessoas_docentes])

In [57]:
pessoas

,AN_BASE,ID_PESSOA_HASH,NM,AN_NASCIMENTO,NM_PAIS_NACIONALIDADE,DS_TIPO_NACIONALIDADE,TP_SEXO
0,2021,3e904591534275abcc843da1d0a3614e1a7b6d302d236f...,BRENDO ARAUJO GOMES,1994,BRASIL,BRASILEIRO,M
1,2021,bb5f299988d99383b4a6e30e92924031638eb4fe3e6b98...,CLAUDIA BENITEZ LOGELO,1973,BRASIL,BRASILEIRO,F
2,2021,7c408169ed58a44e99d6ecfae493feaa0c4deae7ba3e22...,FRANCIANE PIMENTEL MELO,1974,BRASIL,BRASILEIRO,F
3,2021,865ea08e85fb688cf728650403234496c664815fa1d7e8...,GABRIELA MONTEZ HOLANDA DA SILVA,1990,BRASIL,BRASILEIRO,F
4,2021,b1048726933afef278ba1ee518e0647781d52241cfb241...,DANIELLE BRODA DE VASCONCELLOS,1991,BRASIL,BRASILEIRO,F
...,...,...,...,...,...,...,...
42360,2018,e6a847b74e40d7a91afd475c69215fd5be5577d0fe3e89...,MARILEIA FRANCO MARINHO INOUE,1960,BRASIL,BRASILEIRO,F
42361,2018,455d3e34df1bcd706dc8da5372e433ee35f4227e3f6977...,JORGE PAES BARRETO MARCONDES DE SOUZA,1957,BRASIL,BRASILEIRO,M
42362,2018,d464d11421ef42f1617b719cc8fe2945f07d034eea29ac...,CARLOS MAGLUTA,1958,BRASIL,BRASILEIRO,M
42363,2018,6cb8b1a40f58aed41b50df80b7519b1f088dcff84671f0...,MARCOS DANTAS LOUREIRO,1948,BRASIL,BRASILEIRO,M


In [58]:
#Após inspeção visual do dataset, averiguamos que há inconsistências com relação a campos como gênero e nacionalidade ao longo dos anos
#Erros de preenchimento podem ter ocorrido, mas eventos como naturalização e mudança de nome social também parecem desempenhar um papel importante
print(f'Um único id_pessoa pode aparecer com mais de uma nacionalidade? {pessoas.drop_duplicates(['ID_PESSOA_HASH','NM_PAIS_NACIONALIDADE'])['ID_PESSOA_HASH'].duplicated().any()}')
print(f'Um único id_pessoa pode aparecer com mais de um gênero? {pessoas.drop_duplicates(['ID_PESSOA_HASH','TP_SEXO'])['ID_PESSOA_HASH'].duplicated().any()}')

Um único id_pessoa pode aparecer com mais de uma nacionalidade? True
Um único id_pessoa pode aparecer com mais de um gênero? True


In [59]:
#Independentemente das inconsistências terem sido derivadas de erros de preenchimento ou não, uma abordagem adequada para essa tabela seria
#manter apenas os registros do último ano amostrado por pessoa e permitir a atualização com base no ano mais recente com o django ORM depois (update_or_create()).
#Assim, teremos sempre as informações mais recentes daquela pessoa.

pessoas_ultimo_ano = pessoas.sort_values('AN_BASE', ascending=False).drop_duplicates('ID_PESSOA_HASH') #Organiza a tabela por ano (descendente) e mantém apenas a primeira ocorrência de cada ID_PESSOA

In [60]:
#Dando uma olhada nos dados
pessoas_ultimo_ano

,AN_BASE,ID_PESSOA_HASH,NM,AN_NASCIMENTO,NM_PAIS_NACIONALIDADE,DS_TIPO_NACIONALIDADE,TP_SEXO
158391,2024,1887ea771adfd7ef0f6b5069b7e046d4dfde53bfb9b536...,ALINE TEIXEIRA DE CARVALHO,1981,BRASIL,BRASILEIRO,F
158392,2024,5dc0b39a0bcc5e389541c5f80f45b187011e82f25956ab...,CHRISTIANE STEFANY BRAZAO PINTO,1990,BRASIL,BRASILEIRO,F
158393,2024,771a722568cda665f0952ecd62f336e369e223b08baaa8...,DEBORAH ROBERTA NUNEZ NASCIMENTO LOPES,1976,BRASIL,BRASILEIRO,F
158394,2024,61b88314dc763383865e4d28f8f2db61db658b4e3b3f2b...,HENRIQUE GERVASIO NEVES RONDINELLI,1992,BRASIL,BRASILEIRO,M
158395,2024,a40549afafadd60f173defbdf5d6be64cb915a826f49e2...,BEATRIZ RODRIGUES FERNANDES PEDERSANE,1997,BRASIL,BRASILEIRO,F
...,...,...,...,...,...,...,...
28055,2013,46022b52b00421f7ef014a9a4b85449ccacd86d283dbab...,LUCIANA RODRIGUES BARRETO LOPES,1979,BRASIL,BRASILEIRO,F
28087,2013,d85dc8240e67ddeabc9d7307b911baa77c7baa065cb84a...,ANA CRISTINA BOMFIM PEIXOTO,1986,BRASIL,BRASILEIRO,F
18992,2013,565a8cc47673c02c7074c71a61f3d43beb97f05828a630...,VICTOR TEIXEIRA DE MELO MAYRINK,1988,BRASIL,BRASILEIRO,M
18986,2013,b2d787844d5fb2a6d7ac9251bee18ebe7e8fdaaaa5c9ed...,LINCOLN CHAVES RIBEIRO,1985,BRASIL,BRASILEIRO,M


In [61]:
#Removendo colunas desnecessárias para a database
pessoas_final = pessoas_ultimo_ano.drop(columns=['AN_BASE','NM'])

#Limpando possíveis valores não informados do nome do país
pessoas_final['NM_PAIS_NACIONALIDADE'] = pessoas_final['NM_PAIS_NACIONALIDADE'].replace('NÃO INFORMADO', pd.NA)

In [62]:
#Salvando dataframe final
pessoas_final.to_csv(f'{processed_dir}/pessoas.csv', index=False)

### Programas

In [63]:
#Importando df filtrada
programas = pd.read_csv(f'{filtered_dir}/programas.csv')
programas

,AN_BASE,CD_PROGRAMA_IES,NM_PROGRAMA_IES,NM_GRANDE_AREA_CONHECIMENTO,NM_AREA_CONHECIMENTO,NM_GRAU_PROGRAMA,CD_CONCEITO_PROGRAMA,ANO_INICIO_PROGRAMA,AN_INICIO_PROGRAMA,AN_INICIO_CURSO,IN_REDE,DS_SITUACAO_PROGRAMA,CD_AREA_AVALIACAO,NM_AREA_AVALIACAO,NM_MODALIDADE_PROGRAMA,SG_ENTIDADE_ENSINO,SG_ENTIDADE_ENSINO_REDE
0,2017,23001011069P5,LETRAS,"LINGÜÍSTICA, LETRAS E ARTES",LETRAS,MESTRADO PROFISSIONAL,4,NaN,2013.0,2013,SIM,EM FUNCIONAMENTO,41,LINGUÍSTICA E LITERATURA,PROFISSIONAL,UFRN,USP;UPE;UNIOESTE;UNIMONTES;UNIFESSPA;UNESP-ARA...
1,2017,31102000001P6,PROFNIT - PROPRIEDADE INTELECTUAL E TRANSFERÊN...,CIÊNCIAS SOCIAIS APLICADAS,ADMINISTRAÇÃO,MESTRADO PROFISSIONAL,4,NaN,2016.0,2016,SIM,EM FUNCIONAMENTO,27,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",PROFISSIONAL,UFAL,UNIVASF;UNIFESSPA;UNIFAP;UNICENTRO;UNEMAT;UNB;...
2,2017,31001017005P0,ESTATÍSTICA,CIÊNCIAS EXATAS E DA TERRA,PROBABILIDADE E ESTATÍSTICA,MESTRADO/DOUTORADO,5,NaN,1981.0,1981/2001,NÃO,EM FUNCIONAMENTO,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,ACADÊMICO,UFRJ,NaN
3,2017,31001017162P8,SAÚDE PERINATAL,CIÊNCIAS DA SAÚDE,MEDICINA,MESTRADO PROFISSIONAL,3,NaN,2015.0,2015,NÃO,EM FUNCIONAMENTO,16,MEDICINA II,PROFISSIONAL,UFRJ,NaN
4,2017,31001017158P0,ENGENHARIA DA NANOTECNOLOGIA,ENGENHARIAS,ENGENHARIA DE MATERIAIS E METALÚRGICA,MESTRADO/DOUTORADO,4,NaN,2014.0,2014/2014,NÃO,EM FUNCIONAMENTO,12,ENGENHARIAS II,ACADÊMICO,UFRJ,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1507,2014,31001017158P0,ENGENHARIA DA NANOTECNOLOGIA,ENGENHARIAS,ENGENHARIA DE MATERIAIS E METALÚRGICA,MESTRADO/DOUTORADO,5,2014.0,NaN,2014/2014,NÃO,EM FUNCIONAMENTO,12,ENGENHARIAS II,ACADÊMICO,UFRJ,NaN
1508,2014,31075010001P2,MATEMÁTICA EM REDE NACIONAL,CIÊNCIAS EXATAS E DA TERRA,MATEMÁTICA,MESTRADO PROFISSIONAL,5,2011.0,NaN,2011,SIM,EM FUNCIONAMENTO,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,PROFISSIONAL,SBM,UTFPR;USP/SC;USP/RP;USP;UNIVASF;UNIRIO;UNIR;UN...
1509,2014,33147019001P2,MULTICÊNTRICO EM CIÊNCIAS FISIOLÓGICAS,CIÊNCIAS BIOLÓGICAS,FISIOLOGIA,MESTRADO/DOUTORADO,4,2009.0,NaN,2009/2009,SIM,EM FUNCIONAMENTO,8,CIÊNCIAS BIOLÓGICAS II,ACADÊMICO,SBFIS,USP;UNIFAL;UNESP/ARAÇ;UFVJM;UFSC;UFRRJ;UFRJ;UF...
1510,2014,33283010001P5,ENSINO DE FÍSICA - PROFIS,CIÊNCIAS EXATAS E DA TERRA,FÍSICA,MESTRADO PROFISSIONAL,4,2013.0,NaN,2013,SIM,EM FUNCIONAMENTO,3,ASTRONOMIA / FÍSICA,PROFISSIONAL,SBF,UTFPR;URCA;UNIVASF;UNIRIO;UNIR;UNIFESSPA;UNIFA...


In [64]:
programas[programas['NM_PROGRAMA_IES'] == 'EDUCAÇÃO FÍSICA']

,AN_BASE,CD_PROGRAMA_IES,NM_PROGRAMA_IES,NM_GRANDE_AREA_CONHECIMENTO,NM_AREA_CONHECIMENTO,NM_GRAU_PROGRAMA,CD_CONCEITO_PROGRAMA,ANO_INICIO_PROGRAMA,AN_INICIO_PROGRAMA,AN_INICIO_CURSO,IN_REDE,DS_SITUACAO_PROGRAMA,CD_AREA_AVALIACAO,NM_AREA_AVALIACAO,NM_MODALIDADE_PROGRAMA,SG_ENTIDADE_ENSINO,SG_ENTIDADE_ENSINO_REDE
11,2017,31001017131P5,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO/DOUTORADO,3,NaN,2009.0,2009/2016,NÃO,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,ACADÊMICO,UFRJ,NaN
212,2013,31001017131P5,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO/DOUTORADO,3,2009.0,NaN,2009/2016,NÃO,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,ACADÊMICO,UFRJ,NaN
352,2019,31001017131P5,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO/DOUTORADO,3,NaN,2009.0,2009/2016,NÃO,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,ACADÊMICO,UFRJ,NaN
368,2022,31001017131P5,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO/DOUTORADO,4,NaN,2009.0,2009/2016,NÃO,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,ACADÊMICO,UFRJ,NaN
495,2022,33004137068P8,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO PROFISSIONAL,3,NaN,2018.0,2018,SIM,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,PROFISSIONAL,UNESP-PP,UPE;UNIOESTE;UNIMONTES;UNIJUÍ;UNESP-RC;UNESP-B...
605,2024,31001017131P5,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO/DOUTORADO,4,NaN,2009.0,2009/2016,NÃO,EM FUNCIONAMENTO,21,"EDUCAÇÃO FÍSICA, FISIOTERAPIA, FONOAUDIOLOGIA ...",ACADÊMICO,UFRJ,NaN
704,2021,31001017131P5,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO/DOUTORADO,4,NaN,2009.0,2009/2016,NÃO,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,ACADÊMICO,UFRJ,NaN
762,2021,33004137068P8,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO PROFISSIONAL,3,NaN,2018.0,2018,SIM,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,PROFISSIONAL,UNESP-PP,UPE;UNIOESTE;UNIMONTES;UNIJUÍ;UNESP-RC;UNESP-B...
881,2023,31001017131P5,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO/DOUTORADO,4,NaN,2009.0,2009/2016,NÃO,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,ACADÊMICO,UFRJ,NaN
895,2023,33004137068P8,EDUCAÇÃO FÍSICA,CIÊNCIAS DA SAÚDE,EDUCAÇÃO FÍSICA,MESTRADO PROFISSIONAL,3,NaN,2018.0,2018,SIM,EM FUNCIONAMENTO,21,EDUCAÇÃO FÍSICA,PROFISSIONAL,UNESP-PP,UPE;UNIOESTE;UNIMONTES;UNIJUÍ;UNESP-RC;UNESP-B...


In [65]:
#Gerando colunas com os hashes baseados nos ids originais
programas['ID_PROGRAMA_HASH'] = converter_ids_para_hashes(programas['CD_PROGRAMA_IES'])
programas[['CD_PROGRAMA_IES', 'ID_PROGRAMA_HASH']]

,CD_PROGRAMA_IES,ID_PROGRAMA_HASH
0,23001011069P5,63ef38d248cf563b31fb56b5d06b3de282c84a0f66ae11...
1,31102000001P6,292e3ce8eb45da4cf30b4b0ef292b826288c2dfbb63d9a...
2,31001017005P0,99c872ccae8d68d8aca1cef68472e2b466865fdf6d392d...
3,31001017162P8,6b466c49c7610cf4c6f346f1da3a2652bb22ab143157ea...
4,31001017158P0,a0856cb720d2d79c28250d1b3666b1938d2f761a0a3282...
...,...,...
1507,31001017158P0,a0856cb720d2d79c28250d1b3666b1938d2f761a0a3282...
1508,31075010001P2,b90a3796b044f9cb9cda4f71b512563a212baad254a612...
1509,33147019001P2,12fead5876810c0ebeceb98b87a637e9e51f37e294efaf...
1510,33283010001P5,79147cb843facf59c051c83de2c65b0322d5f20676adb0...


In [66]:
#Removendo o SG_ENTIDADE_ENSINO (todos os programas são da UFRJ, afinal)
programas = programas.drop(columns=['SG_ENTIDADE_ENSINO'])

In [67]:
#Checando valores do campo 'IN_REDE'
programas['IN_REDE'].unique()

array(['SIM', 'NÃO'], dtype=object)

In [68]:
#Convertendo valores para booleans
mapeamento_booleano = {'SIM': True, 'NÃO': False}
programas['IN_REDE'] = programas['IN_REDE'].map(mapeamento_booleano)

In [69]:
programas['IN_REDE'].unique()

array([ True, False])

In [70]:
#Checando conceitos atribuídos às PPGs
programas['CD_CONCEITO_PROGRAMA'].unique()

array(['4', '5', '3', '6', '7', 'A', '1'], dtype=object)

In [71]:
#Convertendo conceito 'A' (provavelmente 'Ausente') para 0
programas['CD_CONCEITO_PROGRAMA'] = programas['CD_CONCEITO_PROGRAMA'].replace('A', 0).astype('int') 

In [72]:
programas['CD_CONCEITO_PROGRAMA'].unique()

array([4, 5, 3, 6, 7, 0, 1])

In [73]:
#Checando colunas ANO_INICIO_PROGRAMA e AN_INICIO_PROGRAMA 
print(programas[['ANO_INICIO_PROGRAMA', 'AN_INICIO_PROGRAMA']])

      ANO_INICIO_PROGRAMA  AN_INICIO_PROGRAMA
0                     NaN              2013.0
1                     NaN              2016.0
2                     NaN              1981.0
3                     NaN              2015.0
4                     NaN              2014.0
...                   ...                 ...
1507               2014.0                 NaN
1508               2011.0                 NaN
1509               2009.0                 NaN
1510               2013.0                 NaN
1511               2014.0                 NaN

[1512 rows x 2 columns]


In [74]:
#Fundindo colunas ANO_INICIO_PROGRAMA e AN_INICIO_PROGRAMA 
#'AN_INICIO_PROGRAMA' convertido para int para evitar a permanência de floats graças aos nan anteriores
programas['AN_INICIO_PROGRAMA'] = programas['AN_INICIO_PROGRAMA'].fillna(programas['ANO_INICIO_PROGRAMA']).astype('int') 

In [75]:
print(programas[['ANO_INICIO_PROGRAMA', 'AN_INICIO_PROGRAMA']])

      ANO_INICIO_PROGRAMA  AN_INICIO_PROGRAMA
0                     NaN                2013
1                     NaN                2016
2                     NaN                1981
3                     NaN                2015
4                     NaN                2014
...                   ...                 ...
1507               2014.0                2014
1508               2011.0                2011
1509               2009.0                2009
1510               2013.0                2013
1511               2014.0                2014

[1512 rows x 2 columns]


In [76]:
#Removendo a coluna 'ANO_INICIO_PROGRAMA'
programas = programas.drop(columns=['ANO_INICIO_PROGRAMA'])

In [77]:
#Visualizando a tabela inteira
programas

,AN_BASE,CD_PROGRAMA_IES,NM_PROGRAMA_IES,NM_GRANDE_AREA_CONHECIMENTO,NM_AREA_CONHECIMENTO,NM_GRAU_PROGRAMA,CD_CONCEITO_PROGRAMA,AN_INICIO_PROGRAMA,AN_INICIO_CURSO,IN_REDE,DS_SITUACAO_PROGRAMA,CD_AREA_AVALIACAO,NM_AREA_AVALIACAO,NM_MODALIDADE_PROGRAMA,SG_ENTIDADE_ENSINO_REDE,ID_PROGRAMA_HASH
0,2017,23001011069P5,LETRAS,"LINGÜÍSTICA, LETRAS E ARTES",LETRAS,MESTRADO PROFISSIONAL,4,2013,2013,True,EM FUNCIONAMENTO,41,LINGUÍSTICA E LITERATURA,PROFISSIONAL,USP;UPE;UNIOESTE;UNIMONTES;UNIFESSPA;UNESP-ARA...,63ef38d248cf563b31fb56b5d06b3de282c84a0f66ae11...
1,2017,31102000001P6,PROFNIT - PROPRIEDADE INTELECTUAL E TRANSFERÊN...,CIÊNCIAS SOCIAIS APLICADAS,ADMINISTRAÇÃO,MESTRADO PROFISSIONAL,4,2016,2016,True,EM FUNCIONAMENTO,27,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",PROFISSIONAL,UNIVASF;UNIFESSPA;UNIFAP;UNICENTRO;UNEMAT;UNB;...,292e3ce8eb45da4cf30b4b0ef292b826288c2dfbb63d9a...
2,2017,31001017005P0,ESTATÍSTICA,CIÊNCIAS EXATAS E DA TERRA,PROBABILIDADE E ESTATÍSTICA,MESTRADO/DOUTORADO,5,1981,1981/2001,False,EM FUNCIONAMENTO,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,ACADÊMICO,NaN,99c872ccae8d68d8aca1cef68472e2b466865fdf6d392d...
3,2017,31001017162P8,SAÚDE PERINATAL,CIÊNCIAS DA SAÚDE,MEDICINA,MESTRADO PROFISSIONAL,3,2015,2015,False,EM FUNCIONAMENTO,16,MEDICINA II,PROFISSIONAL,NaN,6b466c49c7610cf4c6f346f1da3a2652bb22ab143157ea...
4,2017,31001017158P0,ENGENHARIA DA NANOTECNOLOGIA,ENGENHARIAS,ENGENHARIA DE MATERIAIS E METALÚRGICA,MESTRADO/DOUTORADO,4,2014,2014/2014,False,EM FUNCIONAMENTO,12,ENGENHARIAS II,ACADÊMICO,NaN,a0856cb720d2d79c28250d1b3666b1938d2f761a0a3282...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1507,2014,31001017158P0,ENGENHARIA DA NANOTECNOLOGIA,ENGENHARIAS,ENGENHARIA DE MATERIAIS E METALÚRGICA,MESTRADO/DOUTORADO,5,2014,2014/2014,False,EM FUNCIONAMENTO,12,ENGENHARIAS II,ACADÊMICO,NaN,a0856cb720d2d79c28250d1b3666b1938d2f761a0a3282...
1508,2014,31075010001P2,MATEMÁTICA EM REDE NACIONAL,CIÊNCIAS EXATAS E DA TERRA,MATEMÁTICA,MESTRADO PROFISSIONAL,5,2011,2011,True,EM FUNCIONAMENTO,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,PROFISSIONAL,UTFPR;USP/SC;USP/RP;USP;UNIVASF;UNIRIO;UNIR;UN...,b90a3796b044f9cb9cda4f71b512563a212baad254a612...
1509,2014,33147019001P2,MULTICÊNTRICO EM CIÊNCIAS FISIOLÓGICAS,CIÊNCIAS BIOLÓGICAS,FISIOLOGIA,MESTRADO/DOUTORADO,4,2009,2009/2009,True,EM FUNCIONAMENTO,8,CIÊNCIAS BIOLÓGICAS II,ACADÊMICO,USP;UNIFAL;UNESP/ARAÇ;UFVJM;UFSC;UFRRJ;UFRJ;UF...,12fead5876810c0ebeceb98b87a637e9e51f37e294efaf...
1510,2014,33283010001P5,ENSINO DE FÍSICA - PROFIS,CIÊNCIAS EXATAS E DA TERRA,FÍSICA,MESTRADO PROFISSIONAL,4,2013,2013,True,EM FUNCIONAMENTO,3,ASTRONOMIA / FÍSICA,PROFISSIONAL,UTFPR;URCA;UNIVASF;UNIRIO;UNIR;UNIFESSPA;UNIFA...,79147cb843facf59c051c83de2c65b0322d5f20676adb0...


In [78]:
#A tabela 'pessoas' deve manter apenas dados que não podem mudar para um dado programa ao longo dos anos
#Dentre os campos utilizados no painel, espera-se que isso seja verdade apenas para o campo 'AN_INICIO_PROGRAMA'
#NOTA: Há programas que apresentam variação para esse campo, mas acreditamos que seja por conta de erros de preenchimento
#Logo, assim como para pessoas, nós iremos manter na tabela 'programas' apenas a informação do último ano amostrado (mais atualizado) 
programas_final_cols = ['AN_BASE', 'ID_PROGRAMA_HASH', 'AN_INICIO_PROGRAMA']
programas_final = programas[programas_final_cols].sort_values( #Seleciona colunas de interesse e ordena a tabela por ano (descendente)
                                                    'AN_BASE', 
                                                    ascending=False
                                                    ).drop_duplicates( #Mantém apenas a primeira ocorrência de 'CD_PROGRAMA_IES'
                                                    'ID_PROGRAMA_HASH'
                                                    ).drop(columns=['AN_BASE']) #Remove a coluna AN_BASE, que só foi necessária aqui para ordenar a tabela

In [79]:
#Vendo a tabela final
programas_final

,ID_PROGRAMA_HASH,AN_INICIO_PROGRAMA
532,547dcb7a1aec2acb5a49675d049affe331a4b5eee4a187...,2003
531,8a1c0392f13b5c36830864344e91a41d16434bf3590000...,2006
530,7692998a4083fd27b277d59c3535f32c8947729b6cf2c8...,2008
529,b86fa5c69b8beefdde1f219e64f7c5f19f1a1662f241d1...,2014
632,867609cf3145d4b8a8e4dc4a88679b5402afb79bd93698...,2009
...,...,...
514,4924c27c04937081e2d8e484e5355f739406b2488ed402...,1985
513,ab80bbafe5303cf70c1d6ae4dbb054e86a1b450a4cc067...,2016
512,5707994a842644c619e189e5e43bbc1f5c5d229993baaf...,1985
528,d5943a9242f3da993102d3cab1023291717f00c7be71a8...,1963


In [80]:
#Salvando o dataframe final em um csv
programas_final.to_csv(f'{processed_dir}/programas.csv', index=False)

### Ano_programas

In [81]:
#Pode ser que programas apresentem o mesmo nome no último ano amostrado (nome atual do programa) 
# É o caso dos programas de Educacao Fisica (um em rede (profissional), o outro não (acadêmico))
#Logo, precisamos modificá-los para que o usuário do painel possa identificar cada programa
#Esse procedimento pode gerar valores órfãos nessa tabela, mas mudanças na base da CAPES tbm. Logo, o ideal é um wipe do db antes da atualização
#Talvez só adicionar nome_programa (modalidade_programa) no painel de pós-graduação seja o bastante para lidar com esses nomes duplicados, já que aparentemente isso de repetir o nome só acontece quando há um programa com modalidades distintas
def renomear_programas_mesmo_nome(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ultimo_ano = df["AN_BASE"].max()

    # Identificar nomes duplicados no último ano
    dup_names = (
        df.loc[df["AN_BASE"] == ultimo_ano, "NM_PROGRAMA_IES"]
        .value_counts()
        .loc[lambda x: x > 1]
        .index
    )

    for nome in dup_names:
        # Seleciona apenas os programas que, no último ano, têm esse nome
        codigos = (
            df.loc[(df["AN_BASE"] == ultimo_ano) & (df["NM_PROGRAMA_IES"] == nome),
                   ["CD_PROGRAMA_IES", "AN_INICIO_PROGRAMA"]]
            .drop_duplicates()
        )

        # Ordena por AN_INICIO_PROGRAMA (NA no final) - Programas criados antes recebem o primeiro numero
        codigos = codigos.sort_values(
            by=["AN_INICIO_PROGRAMA"],
            key=lambda x: x.fillna(999999).astype(int)
        ).reset_index(drop=True)

        # Criar mapeamento CD -> Nome numerado (para aquele nome específico)
        mapping = {
            row.CD_PROGRAMA_IES: f"{nome} ({i+1})"
            for i, row in codigos.iterrows()
        }

        # Substitui APENAS nas linhas que têm exatamente esse nome (preservar nomes anteriores dos programas)
        mask = df["NM_PROGRAMA_IES"] == nome
        df.loc[mask, "NM_PROGRAMA_IES"] = df.loc[mask, "CD_PROGRAMA_IES"].map(mapping)

    return df


####Teste da função
#data = [
#    {"AN_BASE": 2020, "CD_PROGRAMA_IES": "A1", "NM_PROGRAMA_IES": "PROG_DUP", "AN_INICIO_PROGRAMA": 2010},
#    {"AN_BASE": 2020, "CD_PROGRAMA_IES": "A2", "NM_PROGRAMA_IES": "PROG_DUP", "AN_INICIO_PROGRAMA": 2015},
#    {"AN_BASE": 2020, "CD_PROGRAMA_IES": "A3", "NM_PROGRAMA_IES": "PROG_DUP", "AN_INICIO_PROGRAMA": None},
#    {"AN_BASE": 2018, "CD_PROGRAMA_IES": "A1", "NM_PROGRAMA_IES": "PROG_DUP", "AN_INICIO_PROGRAMA": 2010},
#    {"AN_BASE": 2018, "CD_PROGRAMA_IES": "A2", "NM_PROGRAMA_IES": "PROG_DUP", "AN_INICIO_PROGRAMA": 2015},
#    {"AN_BASE": 2017, "CD_PROGRAMA_IES": "A2", "NM_PROGRAMA_IES": "PROG_DUP_NOME_ANTIGO", "AN_INICIO_PROGRAMA": 2015},
#    {"AN_BASE": 2016, "CD_PROGRAMA_IES": "A2", "NM_PROGRAMA_IES": "PROG_DUP", "AN_INICIO_PROGRAMA": 2015},
#    {"AN_BASE": 2018, "CD_PROGRAMA_IES": "A3", "NM_PROGRAMA_IES": "PROG_DUP", "AN_INICIO_PROGRAMA": None},
#    {"AN_BASE": 2017, "CD_PROGRAMA_IES": "A3", "NM_PROGRAMA_IES": "PROG_DUP_NOME_ANTIGO", "AN_INICIO_PROGRAMA": None},
#    {"AN_BASE": 2020, "CD_PROGRAMA_IES": "B1", "NM_PROGRAMA_IES": "PROG_UNICO", "AN_INICIO_PROGRAMA": 2005},
#    {"AN_BASE": 2018, "CD_PROGRAMA_IES": "B1", "NM_PROGRAMA_IES": "PROG_UNICO", "AN_INICIO_PROGRAMA": 2005},
#    {"AN_BASE": 2020, "CD_PROGRAMA_IES": "C1", "NM_PROGRAMA_IES": "PROG_DUP2", "AN_INICIO_PROGRAMA": 2012},
#    {"AN_BASE": 2020, "CD_PROGRAMA_IES": "C2", "NM_PROGRAMA_IES": "PROG_DUP2", "AN_INICIO_PROGRAMA": 2012},
#    {"AN_BASE": 2019, "CD_PROGRAMA_IES": "C1", "NM_PROGRAMA_IES": "PROG_DUP2", "AN_INICIO_PROGRAMA": 2012},
#    {"AN_BASE": 2019, "CD_PROGRAMA_IES": "C2", "NM_PROGRAMA_IES": "PROG_DUP2", "AN_INICIO_PROGRAMA": 2012},
#]
#
#df_test = pd.DataFrame(data)
#
#teste = renomear_programas_mesmo_nome(df_test)
#
#teste

In [82]:
programas_renomeados = renomear_programas_mesmo_nome(programas)

In [83]:
#Basicamente, aqui iremos apenas definir as colunas que podem mudar ao longo do tempo (que são a maioria) e extraí-las para obter o dataframe final
ano_programas_cols = ['AN_BASE','ID_PROGRAMA_HASH', 'NM_PROGRAMA_IES', 'CD_CONCEITO_PROGRAMA', 'IN_REDE', 'DS_SITUACAO_PROGRAMA', 
                      'NM_GRANDE_AREA_CONHECIMENTO', 'NM_AREA_CONHECIMENTO', 'CD_AREA_AVALIACAO', 'NM_AREA_AVALIACAO', 'NM_MODALIDADE_PROGRAMA']
ano_programas = programas_renomeados[ano_programas_cols].copy()

In [84]:
#Olhando a tabela completa
ano_programas

,AN_BASE,ID_PROGRAMA_HASH,NM_PROGRAMA_IES,CD_CONCEITO_PROGRAMA,IN_REDE,DS_SITUACAO_PROGRAMA,NM_GRANDE_AREA_CONHECIMENTO,NM_AREA_CONHECIMENTO,CD_AREA_AVALIACAO,NM_AREA_AVALIACAO,NM_MODALIDADE_PROGRAMA
0,2017,63ef38d248cf563b31fb56b5d06b3de282c84a0f66ae11...,LETRAS,4,True,EM FUNCIONAMENTO,"LINGÜÍSTICA, LETRAS E ARTES",LETRAS,41,LINGUÍSTICA E LITERATURA,PROFISSIONAL
1,2017,292e3ce8eb45da4cf30b4b0ef292b826288c2dfbb63d9a...,PROFNIT - PROPRIEDADE INTELECTUAL E TRANSFERÊN...,4,True,EM FUNCIONAMENTO,CIÊNCIAS SOCIAIS APLICADAS,ADMINISTRAÇÃO,27,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",PROFISSIONAL
2,2017,99c872ccae8d68d8aca1cef68472e2b466865fdf6d392d...,ESTATÍSTICA,5,False,EM FUNCIONAMENTO,CIÊNCIAS EXATAS E DA TERRA,PROBABILIDADE E ESTATÍSTICA,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,ACADÊMICO
3,2017,6b466c49c7610cf4c6f346f1da3a2652bb22ab143157ea...,SAÚDE PERINATAL,3,False,EM FUNCIONAMENTO,CIÊNCIAS DA SAÚDE,MEDICINA,16,MEDICINA II,PROFISSIONAL
4,2017,a0856cb720d2d79c28250d1b3666b1938d2f761a0a3282...,ENGENHARIA DA NANOTECNOLOGIA,4,False,EM FUNCIONAMENTO,ENGENHARIAS,ENGENHARIA DE MATERIAIS E METALÚRGICA,12,ENGENHARIAS II,ACADÊMICO
...,...,...,...,...,...,...,...,...,...,...,...
1507,2014,a0856cb720d2d79c28250d1b3666b1938d2f761a0a3282...,ENGENHARIA DA NANOTECNOLOGIA,5,False,EM FUNCIONAMENTO,ENGENHARIAS,ENGENHARIA DE MATERIAIS E METALÚRGICA,12,ENGENHARIAS II,ACADÊMICO
1508,2014,b90a3796b044f9cb9cda4f71b512563a212baad254a612...,MATEMÁTICA EM REDE NACIONAL,5,True,EM FUNCIONAMENTO,CIÊNCIAS EXATAS E DA TERRA,MATEMÁTICA,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,PROFISSIONAL
1509,2014,12fead5876810c0ebeceb98b87a637e9e51f37e294efaf...,MULTICÊNTRICO EM CIÊNCIAS FISIOLÓGICAS,4,True,EM FUNCIONAMENTO,CIÊNCIAS BIOLÓGICAS,FISIOLOGIA,8,CIÊNCIAS BIOLÓGICAS II,ACADÊMICO
1510,2014,79147cb843facf59c051c83de2c65b0322d5f20676adb0...,ENSINO DE FÍSICA - PROFIS,4,True,EM FUNCIONAMENTO,CIÊNCIAS EXATAS E DA TERRA,FÍSICA,3,ASTRONOMIA / FÍSICA,PROFISSIONAL


In [85]:
#Adicionando um campo customizado (DS_CONCEITO), que diz o que o conceito CAPES significa
#O conceito zero foi criado aqui para substituir o 'A'
#A descrição dos outros conceitos foi retirada de: https://www.gov.br/capes/pt-br/centrais-de-conteudo/Artigo_18_07_07.pdf

descricao_conceitos = {
    0: 'AUSÊNCIA DE CONCEITO (PPG RECÉM-CRIADO)',
    1: 'CURSO DESCREDENCIADO',
    2: 'CURSO DESCREDENCIADO',
    3: 'REGULAR',
    4: 'BOM',
    5: 'MUITO BOM',
    6: 'EXCELÊNCIA INTERNACIONAL',
    7: 'EXCELÊNCIA INTERNACIONAL'
}

ano_programas['DS_CONCEITO'] = ano_programas['CD_CONCEITO_PROGRAMA'].map(descricao_conceitos).fillna('DESCRIÇÃO INEXISTENTE')

ano_programas[['CD_CONCEITO_PROGRAMA', 'DS_CONCEITO']]

,CD_CONCEITO_PROGRAMA,DS_CONCEITO
0,4,BOM
1,4,BOM
2,5,MUITO BOM
3,3,REGULAR
4,4,BOM
...,...,...
1507,5,MUITO BOM
1508,5,MUITO BOM
1509,4,BOM
1510,4,BOM


In [86]:
#Salvando dataframe ano_programas para arquivo
ano_programas.to_csv(f'{processed_dir}/ano_programas.csv', index=False)

### Cursos

In [87]:
#Agora, será necessário extrair a tabela 'cursos' de dentro da tabela 'programas'
#Lembrando que alguns programas oferecem mestrado e doutorado simultaneamente
# Logo, precisaremos de alguma lógica para lidar com isso
print(f'Valores únicos: {programas['NM_GRAU_PROGRAMA'].unique()}')
print()
print(programas[['NM_GRAU_PROGRAMA', 'AN_INICIO_CURSO']])

Valores únicos: ['MESTRADO PROFISSIONAL' 'MESTRADO/DOUTORADO' 'MESTRADO' 'DOUTORADO'
 'MESTRADO PROFISSIONAL/DOUTORADO PROFISSIONAL']

           NM_GRAU_PROGRAMA AN_INICIO_CURSO
0     MESTRADO PROFISSIONAL            2013
1     MESTRADO PROFISSIONAL            2016
2        MESTRADO/DOUTORADO       1981/2001
3     MESTRADO PROFISSIONAL            2015
4        MESTRADO/DOUTORADO       2014/2014
...                     ...             ...
1507     MESTRADO/DOUTORADO       2014/2014
1508  MESTRADO PROFISSIONAL            2011
1509     MESTRADO/DOUTORADO       2009/2009
1510  MESTRADO PROFISSIONAL            2013
1511     MESTRADO/DOUTORADO       2014/2014

[1512 rows x 2 columns]


In [88]:
def gerador_tabela_cursos(
    df_programas: pd.DataFrame,
    coluna_id_programa: str = 'ID_PROGRAMA_HASH', 
    coluna_grau: str = 'NM_GRAU_PROGRAMA',
    coluna_ano: str = 'AN_INICIO_CURSO'
) -> pd.DataFrame:
    """
    Desagrega programas que possuem graus combinados (ex: 'MESTRADO/DOUTORADO')
    e anos de início correspondentes (ex: '1981/2001') em linhas separadas.

    Após a desagregação, a função retorna um novo DataFrame contendo apenas
    as colunas CD_PROGRAMA_IES (ou o nome fornecido), NM_GRAU_PROGRAMA (ou o nome fornecido),
    e AN_INICIO_CURSO (ou o nome fornecido), com entradas duplicadas removidas.

    Args:
        df_programas (pd.DataFrame): O DataFrame de entrada.
        coluna_id_programa (str): O nome da coluna que contém o código do programa (ex: 'ID_PROGRAMA_HASH').
        coluna_grau (str): O nome da coluna que contém o grau do programa (padrão: 'NM_GRAU_PROGRAMA').
        coluna_ano (str): O nome da coluna que contém o ano de início do curso (padrão: 'AN_INICIO_CURSO').

    Returns:
        pd.DataFrame: Um novo DataFrame com as colunas especificadas, graus desagregados
                      e entradas duplicadas removidas.

    Raises:
        ValueError: Se as colunas especificadas não existirem no DataFrame.
    """
    # Garante que as colunas existem no DataFrame de entrada
    colunas_necessarias = [coluna_id_programa, coluna_grau, coluna_ano]
    for col in colunas_necessarias:
        if col not in df_programas.columns:
            raise ValueError(f"A coluna '{col}' não foi encontrada no DataFrame. Verifique o nome da coluna.")

    # Trabalha em uma cópia para não modificar o DataFrame original
    df_trabalho = df_programas.copy()

    # Identifica as linhas que contêm '/' na coluna de grau especificada
    mascara_combinados = df_trabalho[coluna_grau].str.contains('/', na=False)
    programas_combinados = df_trabalho[mascara_combinados].copy()
    programas_outros_graus = df_trabalho[~mascara_combinados].copy()

    df_novas_linhas = pd.DataFrame() # DataFrame vazio para acumular as novas linhas geradas

    if not programas_combinados.empty:
        # Divide tanto a coluna de graus quanto a de anos, usando os nomes passados
        graus_divididos = programas_combinados[coluna_grau].str.split('/', expand=True)
        anos_divididos = programas_combinados[coluna_ano].str.split('/', expand=True)

        # Itera sobre as possíveis "partes"
        for i in range(max(len(graus_divididos.columns), len(anos_divididos.columns))):
            nome_grau_parte = graus_divididos.get(i)
            ano_curso_parte = anos_divididos.get(i)

            if nome_grau_parte is not None and ano_curso_parte is not None:
                df_parte = programas_combinados.copy()

                df_parte[coluna_grau] = nome_grau_parte
                df_parte[coluna_ano] = ano_curso_parte

                df_novas_linhas = pd.concat([df_novas_linhas, df_parte], ignore_index=True)

        df_novas_linhas = df_novas_linhas.dropna(subset=[coluna_ano])

    # Concatena o DataFrame de programas sem combinação e as novas linhas geradas
    programas_final_completo = pd.concat([programas_outros_graus, df_novas_linhas], ignore_index=True)

    # --- Nova Lógica para selecionar colunas e remover duplicatas ---

    # Seleciona apenas as colunas desejadas
    df_resultado = programas_final_completo[[coluna_id_programa, coluna_grau, coluna_ano]].copy()

    # Opcional: Converte a coluna de ano de início para tipo numérico
    df_resultado[coluna_ano] = pd.to_numeric(
        df_resultado[coluna_ano], errors='coerce'
    ).astype('Int64') # Usando Int64 para aceitar nulos, como discutido anteriormente

    # Remove entradas duplicadas com base nas três colunas
    df_resultado.drop_duplicates(inplace=True)

    return df_resultado

In [89]:
cursos = gerador_tabela_cursos(programas) #Pode usar a tabela inicial, já que ela vai ter todas as informações

In [90]:
cursos

,ID_PROGRAMA_HASH,NM_GRAU_PROGRAMA,AN_INICIO_CURSO
0,63ef38d248cf563b31fb56b5d06b3de282c84a0f66ae11...,MESTRADO PROFISSIONAL,2013
1,292e3ce8eb45da4cf30b4b0ef292b826288c2dfbb63d9a...,MESTRADO PROFISSIONAL,2016
2,6b466c49c7610cf4c6f346f1da3a2652bb22ab143157ea...,MESTRADO PROFISSIONAL,2015
3,b86fa5c69b8beefdde1f219e64f7c5f19f1a1662f241d1...,MESTRADO PROFISSIONAL,2014
4,5eed314407ba5c5e67cacb87155f645499e1d9b378c50f...,MESTRADO PROFISSIONAL,2013
...,...,...,...
1603,0b04c711de9f9375fd02627577fd7f15e8bafeee9b4396...,DOUTORADO,2014
1734,62e506275ebb1d0075a0995a7ec9c751589bac94d8a67a...,DOUTORADO,2019
1804,863f6ff75ec2dc938392a3e71fde971160a6ad1e267ab6...,DOUTORADO,2020
1826,6ee351da7cca90c2a61d57dab9d12b27defc97646ee2bc...,DOUTORADO,2021


In [91]:
#Salvando a tabela 'cursos' para um csv
cursos.to_csv(f'{processed_dir}/cursos.csv', index=False)

### Produção

In [90]:
#Importando df filtrada
producao = pd.read_csv(f'{filtered_dir}/producao.csv')
producao

,AN_BASE,CD_PROGRAMA_IES,NM_PROGRAMA_IES,ID_ADD_PRODUCAO_INTELECTUAL,ID_PESSOA_DOCENTE,ID_PESSOA_DISCENTE,TP_AUTOR,NM_TP_CATEGORIA_DOCENTE,NM_NIVEL_DISCENTE,SG_ENTIDADE_ENSINO
0,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295536,535426.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
1,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295538,141762.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
2,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295539,476439.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
3,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295539,15859.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
4,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295540,539116.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
...,...,...,...,...,...,...,...,...,...,...
194825,2021,31001017174P6,MULTIDISCIPLINAR EM FÍSICA APLICADA,40772987,107116.0,NaN,DOCENTE,COLABORADOR,NaN,UFRJ
194826,2021,33283010001P5,ENSINO DE FÍSICA - PROFIS,40819247,133164.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
194827,2021,33283010001P5,ENSINO DE FÍSICA - PROFIS,40819247,201748.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
194828,2021,33283010001P5,ENSINO DE FÍSICA - PROFIS,40819247,1009673.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ


In [91]:
producao['NM_NIVEL_DISCENTE'].unique()

array([nan, 'MESTRADO', 'DOUTORADO', 'MESTRADO PROFISSIONAL'],
      dtype=object)

In [92]:
#Fundindo colunas ID_PESSOA_DISCENTE e ID_PESSOA_DOCENTE na coluna ID_PESSOA
producao['ID_PESSOA'] = producao['ID_PESSOA_DOCENTE'].fillna(
    producao['ID_PESSOA_DISCENTE'] #Substituinto NAs de ID_PESSOA_DOCENTE pelos valores em ID_PESSOA_DISCENTE
    ).astype(
        'int' #'ID_PESSOA_DOCENTE' convertido para int para evitar a permanência de floats graças aos nan anteriores
    ).rename(
        'ID_PESSOA' #Renomeando ID_PESSOA_DOCENTE para ID_PESSOA
    )  

In [93]:
producao[['ID_PESSOA', 'ID_PESSOA_DISCENTE', 'ID_PESSOA_DOCENTE', 'TP_AUTOR']]

,ID_PESSOA,ID_PESSOA_DISCENTE,ID_PESSOA_DOCENTE,TP_AUTOR
0,535426,NaN,535426.0,DOCENTE
1,141762,NaN,141762.0,DOCENTE
2,476439,NaN,476439.0,DOCENTE
3,15859,NaN,15859.0,DOCENTE
4,539116,NaN,539116.0,DOCENTE
...,...,...,...,...
194825,107116,NaN,107116.0,DOCENTE
194826,133164,NaN,133164.0,DOCENTE
194827,201748,NaN,201748.0,DOCENTE
194828,1009673,NaN,1009673.0,DOCENTE


In [94]:
#Gerando colunas com os hashes baseados nos ids originais
producao['ID_PESSOA_HASH'] = converter_ids_para_hashes(producao['ID_PESSOA'])
producao['ID_PRODUCAO_HASH'] = converter_ids_para_hashes(producao['ID_ADD_PRODUCAO_INTELECTUAL'])
producao['ID_PROGRAMA_HASH'] = converter_ids_para_hashes(producao['CD_PROGRAMA_IES'])
producao[['ID_PESSOA', 'ID_PESSOA_HASH', 'ID_ADD_PRODUCAO_INTELECTUAL', 'ID_PRODUCAO_HASH', 'CD_PROGRAMA_IES', 'ID_PROGRAMA_HASH']]

,ID_PESSOA,ID_PESSOA_HASH,ID_ADD_PRODUCAO_INTELECTUAL,ID_PRODUCAO_HASH,CD_PROGRAMA_IES,ID_PROGRAMA_HASH
0,535426,b1b17db0e7c33b2036e379ab8a8b4d0951c170d380aa6b...,35295536,30816a06cef9de188f63709f90cf0d2c20b80d93023cca...,31001017015P5,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...
1,141762,d9f64101763aed40b85196ca22336ec9be735254a1da2f...,35295538,85c0bab55107cc7557eb6215c18baa0923f86e191978c2...,31001017015P5,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...
2,476439,834ba153f56635a2d2243874df71451d33e68df0226a02...,35295539,20fcafcc6d376758be3e1f62b7c5acddf770b9b634ee7b...,31001017015P5,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...
3,15859,0f5f5e984db2bf328d45419a4b1e264a7e0e5c279daa01...,35295539,20fcafcc6d376758be3e1f62b7c5acddf770b9b634ee7b...,31001017015P5,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...
4,539116,a486cb2acb1ebf40a8256ef7ac429b5f9789411747600e...,35295540,b9652fa208def118f0de53a5ee59a8ece3ef30a43c6278...,31001017015P5,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...
...,...,...,...,...,...,...
194825,107116,a64328f15d55cc2743bff6acb36a585703714265aec720...,40772987,aa933070f8e05a331ba9ffe474111b12ae9dcb1d03e982...,31001017174P6,76da4842693f49d449f259467aa33ccc1636c210346baf...
194826,133164,NaN,40819247,783b8d734b5c7f785f34d525c2ddcc725e3e60c9e5fecf...,33283010001P5,79147cb843facf59c051c83de2c65b0322d5f20676adb0...
194827,201748,a55a52a71300561b3f5959804a117fefc737042c4c4d11...,40819247,783b8d734b5c7f785f34d525c2ddcc725e3e60c9e5fecf...,33283010001P5,79147cb843facf59c051c83de2c65b0322d5f20676adb0...
194828,1009673,e251e113cd55de6faadcc1c6f9f2ec7860fc5b7d2a9103...,40819247,783b8d734b5c7f785f34d525c2ddcc725e3e60c9e5fecf...,33283010001P5,79147cb843facf59c051c83de2c65b0322d5f20676adb0...


In [95]:
#Checando entradas com autores que não estão na tabela 'pessoas'
print(f"{(~producao['ID_PESSOA_HASH'].isin(pessoas['ID_PESSOA_HASH'])).sum()} linhas com autores que não estão entre os docentes ou discentes da UFRJ") 
print(f"{(~producao.drop_duplicates(subset=['ID_PESSOA'])['ID_PESSOA_HASH'].isin(pessoas['ID_PESSOA_HASH'])).sum()} autores que não estão dentre discentes ou docentes da UFRJ")

732 linhas com autores que não estão entre os docentes ou discentes da UFRJ
72 autores que não estão dentre discentes ou docentes da UFRJ


In [96]:
discentes

,AN_BASE,ID_PESSOA,CD_PROGRAMA_IES,NM_DISCENTE,DS_TIPO_NACIONALIDADE_DISCENTE,NM_PAIS_NACIONALIDADE_DISCENTE,AN_NASCIMENTO_DISCENTE,DS_FAIXA_ETARIA,DS_GRAU_ACADEMICO_DISCENTE,ST_INGRESSANTE,NM_SITUACAO_DISCENTE,QT_MES_TITULACAO,SG_ENTIDADE_ENSINO,ID_PESSOA_HASH,ID_PROGRAMA_HASH,TP_SEXO
0,2021,3353229,31001017100P2,BRENDO ARAUJO GOMES,BRASILEIRO,BRASIL,1994,25 A 29 ANOS,DOUTORADO,True,MATRICULADO,0.0,UFRJ,3e904591534275abcc843da1d0a3614e1a7b6d302d236f...,66751c1feedc37dc12807304bbd673e452000cab1b1755...,M
1,2021,26505,31001017033P3,CLAUDIA BENITEZ LOGELO,BRASILEIRO,BRASIL,1973,45 A 49 ANOS,DOUTORADO,False,MATRICULADO,0.0,UFRJ,bb5f299988d99383b4a6e30e92924031638eb4fe3e6b98...,7b75cd7fbdac4855cb781fe648da4adb61620db94c996e...,F
2,2021,1216819,31001017172P3,FRANCIANE PIMENTEL MELO,BRASILEIRO,BRASIL,1974,45 A 49 ANOS,MESTRADO,False,MATRICULADO,0.0,UFRJ,7c408169ed58a44e99d6ecfae493feaa0c4deae7ba3e22...,06e938eaef4be265435f15838c81ba3b2721bbd908dceb...,F
3,2021,794525,31001017020P9,GABRIELA MONTEZ HOLANDA DA SILVA,BRASILEIRO,BRASIL,1990,30 A 34 ANOS,DOUTORADO,False,MATRICULADO,0.0,UFRJ,865ea08e85fb688cf728650403234496c664815fa1d7e8...,2da72851f4776ed03173310560e21621a1e1e0c3626908...,F
4,2021,4485640,31001017134P4,DANIELLE BRODA DE VASCONCELLOS,BRASILEIRO,BRASIL,1991,30 A 34 ANOS,MESTRADO PROFISSIONAL,True,MATRICULADO,0.0,UFRJ,b1048726933afef278ba1ee518e0647781d52241cfb241...,99ea49d14288358fd2e7eb0305d4476a17efb6ecda6227...,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174316,2024,4320878,31001017111P4,LUCIANA ALCANTARA DA SILVA DOS SANTOS,BRASILEIRO,BRASIL,1991,30 A 34 ANOS,DOUTORADO,True,MATRICULADO,0.0,UFRJ,8184223ad852aa9e9e2ef46ee58588b50a9b895ea61867...,6c5c3c7e488b0614df1789a1ebe21262df43cd295ef5e0...,F
174317,2024,4809377,31001017132P1,IGOR DE SOUZA LEAL FIGUEIREDO,BRASILEIRO,BRASIL,1988,35 A 39 ANOS,MESTRADO PROFISSIONAL,True,MATRICULADO,0.0,UFRJ,585688ecb04e3372d981dcfa460f52aee4cd89e02b1974...,ab073f6b50b70dac250c0d722fd922ce718a5039f89102...,M
174318,2024,1891217,31001017028P0,ANDREIA ARENARI DE SIQUEIRA,BRASILEIRO,BRASIL,1995,25 A 29 ANOS,DOUTORADO,False,MATRICULADO,0.0,UFRJ,f9972e72a5efcc8a5361948f6682aa5fcb6580eb0aac32...,f802b104594ad37dd34281e560156b52d1a7b8eb096099...,F
174319,2024,3889748,31001017038P5,GABRIELA MACIEL WAGNER,BRASILEIRO,BRASIL,1993,30 A 34 ANOS,DOUTORADO,True,MATRICULADO,0.0,UFRJ,677df6ab1a736fb27d59ed4c8e14ffa3f989ca84231760...,27848dd22495a7b2b6104f5ab1d6d99c47b305f4f33609...,F


In [97]:
#Removendo autores não presentes na tabela 'pessoas'
producao = producao[producao['ID_PESSOA_HASH'].isin(pessoas['ID_PESSOA_HASH'])]

In [98]:
#Mantendo apenas as colunas de interesse
producao_final = producao[['AN_BASE', 'ID_PROGRAMA_HASH', 'ID_PRODUCAO_HASH', 'ID_PESSOA_HASH', 'TP_AUTOR', 'NM_NIVEL_DISCENTE', 'NM_TP_CATEGORIA_DOCENTE']].convert_dtypes() #convert_dtypes tenta inferir o melhor tipo para cada coluna

In [99]:
producao_final

,AN_BASE,ID_PROGRAMA_HASH,ID_PRODUCAO_HASH,ID_PESSOA_HASH,TP_AUTOR,NM_NIVEL_DISCENTE,NM_TP_CATEGORIA_DOCENTE
0,2022,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...,30816a06cef9de188f63709f90cf0d2c20b80d93023cca...,b1b17db0e7c33b2036e379ab8a8b4d0951c170d380aa6b...,DOCENTE,<NA>,PERMANENTE
1,2022,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...,85c0bab55107cc7557eb6215c18baa0923f86e191978c2...,d9f64101763aed40b85196ca22336ec9be735254a1da2f...,DOCENTE,<NA>,PERMANENTE
2,2022,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...,20fcafcc6d376758be3e1f62b7c5acddf770b9b634ee7b...,834ba153f56635a2d2243874df71451d33e68df0226a02...,DOCENTE,<NA>,PERMANENTE
3,2022,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...,20fcafcc6d376758be3e1f62b7c5acddf770b9b634ee7b...,0f5f5e984db2bf328d45419a4b1e264a7e0e5c279daa01...,DOCENTE,<NA>,PERMANENTE
4,2022,9986fe8f7d21c51b50de942ab0629ea3b21e86560e953c...,b9652fa208def118f0de53a5ee59a8ece3ef30a43c6278...,a486cb2acb1ebf40a8256ef7ac429b5f9789411747600e...,DOCENTE,<NA>,PERMANENTE
...,...,...,...,...,...,...,...
194824,2021,76da4842693f49d449f259467aa33ccc1636c210346baf...,815fd75229eff6bc7b8d77031915b32f1cf3195483054c...,2d15cc654966dc2ee795b4defa45d5eefca4ba52349ee8...,DOCENTE,<NA>,PERMANENTE
194825,2021,76da4842693f49d449f259467aa33ccc1636c210346baf...,aa933070f8e05a331ba9ffe474111b12ae9dcb1d03e982...,a64328f15d55cc2743bff6acb36a585703714265aec720...,DOCENTE,<NA>,COLABORADOR
194827,2021,79147cb843facf59c051c83de2c65b0322d5f20676adb0...,783b8d734b5c7f785f34d525c2ddcc725e3e60c9e5fecf...,a55a52a71300561b3f5959804a117fefc737042c4c4d11...,DOCENTE,<NA>,PERMANENTE
194828,2021,79147cb843facf59c051c83de2c65b0322d5f20676adb0...,783b8d734b5c7f785f34d525c2ddcc725e3e60c9e5fecf...,e251e113cd55de6faadcc1c6f9f2ec7860fc5b7d2a9103...,DOCENTE,<NA>,PERMANENTE


In [100]:
#Salvando dataframe producao_final como csv
producao_final.to_csv(f'{processed_dir}/producao.csv', index=False)

Com os dados processados, agora é possível usá-los para popular o banco de dados do painel de dados de pesquisa e pós-graduação da UFRJ.

In [101]:
#Alguns ID_PROGRAMA_HASH da tabela dos docentes não estão na tabela programas
#Quero checar se isso acontece aqui tbm 

for i in docentes['ID_PROGRAMA_HASH'].to_list():
    if i in programas['ID_PROGRAMA_HASH'].to_list():
        continue
        #print(f'{i} not in programas')
    else:
        print(f'{i} in programas')

In [102]:
for i in discentes['CD_PROGRAMA_IES'].to_list():
    if i in programas['CD_PROGRAMA_IES'].to_list():
        #continue
        print(f'{i} not in programas')
    else:
        continue
        #print(f'{i} in programas')

31001017100P2 not in programas
31001017033P3 not in programas
31001017172P3 not in programas
31001017020P9 not in programas
31001017134P4 not in programas
31001017170P0 not in programas
31001017112P0 not in programas
31001017008P9 not in programas
31001017014P9 not in programas
31001017014P9 not in programas
31001017006P6 not in programas
31001017154P5 not in programas
31001017172P3 not in programas
31001017003P7 not in programas
31001017001P4 not in programas
31001017113P7 not in programas
31001017029P6 not in programas
31001017152P2 not in programas
31001017012P6 not in programas
31001017031P0 not in programas
31001017111P4 not in programas
31001017029P6 not in programas
31001017111P4 not in programas
31001017020P9 not in programas
31001017099P4 not in programas
31001017017P8 not in programas
31001017062P3 not in programas
31001017033P3 not in programas
31075010001P2 not in programas
31001017029P6 not in programas
31001017101P9 not in programas
31001017028P0 not in programas
31001017

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [103]:
ppgs_em_comum = discentes['CD_PROGRAMA_IES'].isin(programas['CD_PROGRAMA_IES'])

In [104]:
elem_em_comum = discentes[ppgs_em_comum]

In [105]:
len(programas['CD_PROGRAMA_IES'].unique())

135

In [106]:
#
len(elem_em_comum['CD_PROGRAMA_IES'].unique())

134

In [107]:
len(discentes['CD_PROGRAMA_IES'].unique())

134

In [108]:
discentes[~discentes['CD_PROGRAMA_IES'].isin(elem_em_comum['CD_PROGRAMA_IES'])]

,AN_BASE,ID_PESSOA,CD_PROGRAMA_IES,NM_DISCENTE,DS_TIPO_NACIONALIDADE_DISCENTE,NM_PAIS_NACIONALIDADE_DISCENTE,AN_NASCIMENTO_DISCENTE,DS_FAIXA_ETARIA,DS_GRAU_ACADEMICO_DISCENTE,ST_INGRESSANTE,NM_SITUACAO_DISCENTE,QT_MES_TITULACAO,SG_ENTIDADE_ENSINO,ID_PESSOA_HASH,ID_PROGRAMA_HASH,TP_SEXO


In [109]:
progs_not_ufrj = discentes[~discentes['CD_PROGRAMA_IES'].isin(elem_em_comum['CD_PROGRAMA_IES'])]['CD_PROGRAMA_IES'].unique()

In [110]:
progs_not_ufrj

array([], dtype=object)

In [ ]:
### AMANHÃ - lidar com o problema de ID_PROGRAMAS que não são da UFRJ

#Primeiro fazer a filtragem dos programas
#Daí, fazer com que tudo que usa CD_PROGRAMA_IES seja filtrado não por universidade, mas sim por ID do programa